Forecasting Algotrithm (Using XGBoost) For Hourly Gas Burns
Summer 2026
Clarissa J. Reynolds
Student Intern - Energy Trading


In [1]:
#Import necessary libraries
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from datetime import timedelta
import os
from pathlib import Path
from sklearn.metrics import mean_squared_error
from mssql_python import connect
from nbdevAuto.functions import * 
import nbdevAuto.functions
import time 

In [2]:
# Database connection
SQL_CONNECTION_STRING = ("Server=HDQv1958;" "Database=allegro;" "Trusted_Connection=yes;" "Encrypt=yes;" "TrustServerCertificate=yes;")

conn = connect(SQL_CONNECTION_STRING)

In [3]:
#Getting site information and Daily gas burn 

def load_burn():

    query = """
SELECT t.trade, p.marketarea, q.begtime, q.energy, q.quantitystatus

FROM trade t, position p, ngquantity q

WHERE p.bepc_strategy = 'Burn' and t.trade = p.trade and t.tradestatus <> 'Void' and p.position = q.position and q.posstatus = 1
AND q.begtime >= DATEADD(year, -2, GETDATE()) AND q.begtime <= GETDATE()

ORDER BY q.begtime"""

    df = pd.read_sql(query, conn)
    #df["datetime"] = pd.to_datetime(df["datetime"])

    return df[["begtime", "marketarea", "energy"]]

gas_daily_df = load_burn()

#gas_daily_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 1.csv", index=False)

C:\Users\a102193\AppData\Local\Temp\ipykernel_37820\1453939605.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [2]:
# Caleb Data Read
gas_daily_df = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 1.csv")

In [3]:
#Adding the columns from the data that we need in the final dataset

#defining market areas as sites
gas_daily_df["marketarea"] = gas_daily_df["marketarea"].str.upper().str.strip()

gas_daily_df = gas_daily_df[
    ~gas_daily_df["marketarea"].isin(["BISON", "COTTAGE GROVE"])]

site_map = {"DEER CREEK": "DCS", "LANARK": "CGS", "LONSOME CREEK": "LCS",  "STATELINE": "PGS", "GROTON": "GGS", "CULBERTSON": "CGS"}

gas_daily_df["site"] = gas_daily_df["marketarea"].replace(site_map)



# create gas day

gas_daily_df["gas_day"] = pd.to_datetime(gas_daily_df["begtime"])


In [4]:

gas_daily_df["gas_day"] = pd.to_datetime(gas_daily_df["gas_day"]).dt.date


In [5]:
# aggregate on gas_day and renaming energy column to daily_gas_burn 
gas_daily_site = (gas_daily_df.groupby(["gas_day", "site"], as_index=False)["energy"].sum().rename(columns={"energy": "daily_gas_burn"}))


In [6]:
gas_daily_site.head()

,gas_day,site,daily_gas_burn
0,2024-07-23,CGS,13523.0
1,2024-07-23,DCS,46474.0
2,2024-07-23,GGS,11036.0
3,2024-07-23,LCS,42282.0
4,2024-07-23,PGS,33933.0


The next two queries are separate since PGS is aggregated by five minute intervals but the other generation sites are aggregated by hour

In [8]:
#Loading generation data for non-PGS sites

def load_generation_nonpgs():

    query = """
    SELECT begtime, loadshape, he1, he2, he3, he4, he5, he6, he7, he8, he9, he10, he11, he12, he13, he14, he15, he16, he17, he18, he19, he20, he21, he22, he23, he24
    FROM dbo.loadshapeprofile
    WHERE begtime >= DATEADD(year, -2, GETDATE()) AND begtime <= GETDATE()
      AND loadshape IN ('WAUE.BEPM.DCS1 - Net Generation',
        'WAUE.BEPM.LCS1 - Net Generation', 'WAUE.BEPM.LCS2 - Net Generation', 'WAUE.BEPM.LCS3 - Net Generation', 'WAUE.BEPM.LCS4 - Net Generation', 'WAUE.BEPM.LCS5 - Net Generation',
        'WAUE.BEPM.LCS6 - Net Generation', 'WAUE.BEPM.GGS1 - Net Generation', 'WAUE.BEPM.GGS2 - Net Generation', 'WAUE.BEPM.CULBERTSON1 - Net Generation')
    """

    return pd.read_sql(query, conn)

df_nonpgs = load_generation_nonpgs()

C:\Users\a102193\AppData\Local\Temp\ipykernel_37820\2149839047.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [ ]:
#df_nonpgs.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 2.csv", index=False)

In [7]:
# Caleb data read
df_nonpgs = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 2.csv")

In [9]:
#Loading generation data for PGS

def load_generation_pgs():

    query = """SELECT begtime, loadshape, SUM(he1)  AS he1, SUM(he2)  AS he2, SUM(he3)  AS he3, SUM(he4)  AS he4, SUM(he5)  AS he5, SUM(he6)  AS he6, SUM(he7)  AS he7,
    SUM(he8)  AS he8, SUM(he9)  AS he9, SUM(he10) AS he10, SUM(he11) AS he11, SUM(he12) AS he12, SUM(he13) AS he13, SUM(he14) AS he14, SUM(he15) AS he15, SUM(he16) AS he16,
    SUM(he17) AS he17, SUM(he18) AS he18, SUM(he19) AS he19, SUM(he20) AS he20, SUM(he21) AS he21, SUM(he22) AS he22, SUM(he23) AS he23, SUM(he24) AS he24

    FROM dbo.loadshapeprofile

    WHERE loadshape LIKE '%PGS%' AND loadshape LIKE '%- Net Generation-5m' AND begtime >= DATEADD(year, -2, GETDATE()) AND begtime <= GETDATE()

    GROUP BY begtime, loadshape

    ORDER BY begtime;"""

    return pd.read_sql(query, conn)

df_pgs = load_generation_pgs()

C:\Users\a102193\AppData\Local\Temp\ipykernel_37820\1754550502.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [ ]:
#df_pgs.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 3.csv", index=False)

In [8]:
# Caleb Data Load
df_pgs = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 3.csv")

In [9]:
#Combining the data sources into one dataframe and mapping multiple generators to their corresponding sites 


df_load_generation = pd.concat([df_nonpgs, df_pgs], ignore_index=True)

# convert date using begtime
df_load_generation["begtime"] = pd.to_datetime(df_load_generation["begtime"])
df_load_generation["date"] = df_load_generation["begtime"].dt.date


# map site
df_load_generation["site"] = "N/A"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("DCS"), "site"] = "DCS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("LCS"), "site"] = "LCS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("PGS"), "site"] = "PGS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("GGS"), "site"] = "GGS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("CULBERTSON"), "site"] = "CGS"

In [10]:
df_load_generation.head()

,begtime,loadshape,he1,he2,he3,he4,he5,he6,he7,he8,...,he17,he18,he19,he20,he21,he22,he23,he24,date,site
0,2024-08-01,WAUE.BEPM.CULBERTSON1 - Net Generation,38.1,38.0,38.1,38.2,38.1,38.3,38.0,38.1,...,61.5,61.5,61.8,61.5,64.6,69.4,76.5,39.4,2024-08-01,CGS
1,2024-08-02,WAUE.BEPM.CULBERTSON1 - Net Generation,38.0,37.6,37.9,38.0,38.2,38.2,38.2,38.4,...,60.6,59.9,59.3,59.3,61.3,61.1,64.8,40.8,2024-08-02,CGS
2,2024-08-03,WAUE.BEPM.CULBERTSON1 - Net Generation,38.1,38.1,38.3,38.0,38.2,38.1,38.2,38.5,...,65.3,64.5,65.2,65.2,69.3,67.2,65.4,39.0,2024-08-03,CGS
3,2024-08-04,WAUE.BEPM.CULBERTSON1 - Net Generation,38.0,38.4,38.2,38.4,38.3,38.1,38.1,38.6,...,66.0,15.1,0.0,0.0,0.0,0.0,0.0,0.0,2024-08-04,CGS
4,2024-08-05,WAUE.BEPM.CULBERTSON1 - Net Generation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-08-05,CGS


In [11]:
# melt function (python) instead of unpivot (SQL) which makes the data in long format instead of wide format

hour_cols = [column for column in df_load_generation.columns if column.startswith("he")]

hourly_df = df_load_generation.melt(id_vars=["begtime", "site", "loadshape"], value_vars=hour_cols, var_name="hour", value_name="hourly_mw")



In [12]:
hourly_df.head()

,begtime,site,loadshape,hour,hourly_mw
0,2024-08-01,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he1,38.1
1,2024-08-02,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he1,38.0
2,2024-08-03,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he1,38.1
3,2024-08-04,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he1,38.0
4,2024-08-05,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he1,0.0


In [13]:
#standardize he to four characters (chronological)
hourly_df['hour'] =  np.where(hourly_df['hour'].str.len()==3, hourly_df['hour'].str[:2] + '0' + hourly_df['hour'].str[2:], hourly_df['hour'])

In [14]:
hourly_df.head()

,begtime,site,loadshape,hour,hourly_mw
0,2024-08-01,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he01,38.1
1,2024-08-02,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he01,38.0
2,2024-08-03,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he01,38.1
3,2024-08-04,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he01,38.0
4,2024-08-05,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he01,0.0


In [15]:
#sort hourly_df
hourly_df =hourly_df.sort_values(by = ['site', 'loadshape', 'begtime', 'hour'], ascending=[False, False, True, True])


In [16]:
# Replace impossible generation spikes with previous hour's value
hourly_df.loc[hourly_df["hourly_mw"] > 400, "hourly_mw"] = np.nan

hourly_df['hourly_mw'] = hourly_df['hourly_mw'].ffill()


In [17]:
print(hourly_df.shape)
hourly_df.head(50000)

(595632, 5)


,begtime,site,loadshape,hour,hourly_mw
12221,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he01,0.0
37039,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he02,0.0
61857,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he03,0.0
86675,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he04,0.0
111493,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he05,0.0
...,...,...,...,...,...
94908,2026-02-14,PGS,WAUE.BEPM.PGS35 - Net Generation-5m,he04,0.0
119726,2026-02-14,PGS,WAUE.BEPM.PGS35 - Net Generation-5m,he05,0.0
144544,2026-02-14,PGS,WAUE.BEPM.PGS35 - Net Generation-5m,he06,0.0
169362,2026-02-14,PGS,WAUE.BEPM.PGS35 - Net Generation-5m,he07,0.0


In [18]:
#9 records that are major outliers for a total of 10501976.59 MW and add back in 207.5 MW from forward fill and filled in daylight savings 


# extract hour number
hourly_df["hour_num"] = hourly_df["hour"].str[-2:].astype(int)

# build datetime
hourly_df["datetime"] = (pd.to_datetime(hourly_df["begtime"]) + pd.to_timedelta(hourly_df["hour_num"] - 0, unit="h"))

 
#hourly_df.head(48)
#hourly_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\Check 7-20 (1).csv", index=False)



In [19]:
# assign gas_day 
hourly_df["gas_day"] = (pd.to_datetime(hourly_df["datetime"]) - pd.Timedelta(hours=10)).dt.date

# extract hour #
hourly_df["hour"] = hourly_df["datetime"].dt.hour


In [20]:
#Reordering columns for visual 
hourly_df = hourly_df[["datetime","gas_day", "hour", "site", "loadshape", "hourly_mw"]]

In [21]:
#Hourly total by site
######################################################################################################################
hourly_site_gen_df = (hourly_df.groupby(["datetime", "gas_day", "hour", "site"], as_index=False)["hourly_mw"].sum().rename(columns={"hourly_mw": "hourly_site_gen_mw"}))

hourly_site_gen_df.head()

,datetime,gas_day,hour,site,hourly_site_gen_mw
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1
1,2024-08-01 01:00:00,2024-07-31,1,DCS,183.0
2,2024-08-01 01:00:00,2024-07-31,1,GGS,0.0
3,2024-08-01 01:00:00,2024-07-31,1,LCS,176.2
4,2024-08-01 01:00:00,2024-07-31,1,PGS,184.6


In [22]:
#Daily total by site
daily_site_gen_df = (hourly_df.groupby(["gas_day", "site"], as_index=False)["hourly_mw"].sum().rename(columns={"hourly_mw": "daily_site_gen_mw"}))
daily_site_gen_df.head()


,gas_day,site,daily_site_gen_mw
0,2024-07-31,CGS,343.4
1,2024-07-31,DCS,1856.0
2,2024-07-31,GGS,66.0
3,2024-07-31,LCS,1566.0
4,2024-07-31,PGS,1509.3


In [23]:
hourly_site_gen_df["gas_day"] = pd.to_datetime( hourly_site_gen_df["gas_day"])

gas_daily_site["gas_day"] = pd.to_datetime(gas_daily_site["gas_day"])

daily_site_gen_df["gas_day"] = pd.to_datetime(daily_site_gen_df["gas_day"])

In [24]:
# Add gas burn data
merged_df = hourly_site_gen_df.merge(gas_daily_site, on=["gas_day", "site"], how="left")
print(merged_df.shape)

(87600, 6)


In [25]:
print(hourly_site_gen_df["gas_day"].min())
print(hourly_site_gen_df["gas_day"].max())

print(gas_daily_site["gas_day"].min())
print(gas_daily_site["gas_day"].max())

print(daily_site_gen_df["gas_day"].min())
print(daily_site_gen_df["gas_day"].max())

2024-07-31 00:00:00
2026-07-31 00:00:00
2024-07-23 00:00:00
2026-07-22 00:00:00
2024-07-31 00:00:00
2026-07-31 00:00:00


In [26]:
test = hourly_site_gen_df.merge(
    gas_daily_site,
    on=["gas_day", "site"],
    how="left"
)

print(test["daily_gas_burn"].notna().sum())
print(len(test))

86565
87600


In [27]:
missing = (
    hourly_site_gen_df[["gas_day", "site"]]
    .drop_duplicates()
    .merge(
        gas_daily_site[["gas_day", "site"]],
        on=["gas_day", "site"],
        how="left",
        indicator=True
    )
)

print(
    missing[missing["_merge"] == "left_only"]
    .head(20)
)

        gas_day site     _merge
3610 2026-07-23  CGS  left_only
3611 2026-07-23  DCS  left_only
3612 2026-07-23  GGS  left_only
3613 2026-07-23  LCS  left_only
3614 2026-07-23  PGS  left_only
3615 2026-07-24  CGS  left_only
3616 2026-07-24  DCS  left_only
3617 2026-07-24  GGS  left_only
3618 2026-07-24  LCS  left_only
3619 2026-07-24  PGS  left_only
3620 2026-07-25  CGS  left_only
3621 2026-07-25  DCS  left_only
3622 2026-07-25  GGS  left_only
3623 2026-07-25  LCS  left_only
3624 2026-07-25  PGS  left_only
3625 2026-07-26  CGS  left_only
3626 2026-07-26  DCS  left_only
3627 2026-07-26  GGS  left_only
3628 2026-07-26  LCS  left_only
3629 2026-07-26  PGS  left_only


In [28]:
merged_df = merged_df.merge(daily_site_gen_df, on =["gas_day", "site"], how="left")
print(merged_df.shape)
merged_df.head(10)


(87600, 7)


,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1,12340.0,343.4
1,2024-08-01 01:00:00,2024-07-31,1,DCS,183.0,43439.0,1856.0
2,2024-08-01 01:00:00,2024-07-31,1,GGS,0.0,9918.0,66.0
3,2024-08-01 01:00:00,2024-07-31,1,LCS,176.2,41713.0,1566.0
4,2024-08-01 01:00:00,2024-07-31,1,PGS,184.6,38470.0,1509.3
5,2024-08-01 02:00:00,2024-07-31,2,CGS,38.0,12340.0,343.4
6,2024-08-01 02:00:00,2024-07-31,2,DCS,207.0,43439.0,1856.0
7,2024-08-01 02:00:00,2024-07-31,2,GGS,0.0,9918.0,66.0
8,2024-08-01 02:00:00,2024-07-31,2,LCS,167.8,41713.0,1566.0
9,2024-08-01 02:00:00,2024-07-31,2,PGS,166.9,38470.0,1509.3


In [29]:
#sort merge_df chronological by site
merged_df = merged_df.sort_values(by=["site", "datetime"])

merged_df.head()

,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1,12340.0,343.4
5,2024-08-01 02:00:00,2024-07-31,2,CGS,38.0,12340.0,343.4
10,2024-08-01 03:00:00,2024-07-31,3,CGS,38.1,12340.0,343.4
15,2024-08-01 04:00:00,2024-07-31,4,CGS,38.2,12340.0,343.4
20,2024-08-01 05:00:00,2024-07-31,5,CGS,38.1,12340.0,343.4


In [30]:
## Caleb Check
merged_df[merged_df["daily_gas_burn"].isna()]

,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw
86565,2026-07-23 10:00:00,2026-07-23,10,CGS,0.0,NaN,0.0
86570,2026-07-23 11:00:00,2026-07-23,11,CGS,0.0,NaN,0.0
86575,2026-07-23 12:00:00,2026-07-23,12,CGS,0.0,NaN,0.0
86580,2026-07-23 13:00:00,2026-07-23,13,CGS,0.0,NaN,0.0
86585,2026-07-23 14:00:00,2026-07-23,14,CGS,0.0,NaN,0.0
...,...,...,...,...,...,...,...
87579,2026-07-31 20:00:00,2026-07-31,20,PGS,0.0,NaN,0.0
87584,2026-07-31 21:00:00,2026-07-31,21,PGS,0.0,NaN,0.0
87589,2026-07-31 22:00:00,2026-07-31,22,PGS,0.0,NaN,0.0
87594,2026-07-31 23:00:00,2026-07-31,23,PGS,0.0,NaN,0.0


In [31]:
##CALEB NOTE: Commenting out next line b/c I don't want to remove data
#merged_df = merged_df[merged_df["daily_gas_burn"].notna()]

print(merged_df.shape)
merged_df.head()

(87600, 7)


,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1,12340.0,343.4
5,2024-08-01 02:00:00,2024-07-31,2,CGS,38.0,12340.0,343.4
10,2024-08-01 03:00:00,2024-07-31,3,CGS,38.1,12340.0,343.4
15,2024-08-01 04:00:00,2024-07-31,4,CGS,38.2,12340.0,343.4
20,2024-08-01 05:00:00,2024-07-31,5,CGS,38.1,12340.0,343.4


In [32]:
print(merged_df.columns.tolist())

['datetime', 'gas_day', 'hour', 'site', 'hourly_site_gen_mw', 'daily_gas_burn', 'daily_site_gen_mw']


In [ ]:
# not dividing by 0
# merged_df["hourly_site_gen_mw"] = (

## Caleb Note: I don't think we should replace 0's with NaNs
merged_df["hourly_site_gen_mw"].replace(0, np.nan)
merged_df["daily_site_gen_mw"].replace(0, np.nan)


,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw,hourly_gas_burn
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1,12340.0,343.4,1369.114735
5,2024-08-01 02:00:00,2024-07-31,2,CGS,38.0,12340.0,343.4,1365.521258
10,2024-08-01 03:00:00,2024-07-31,3,CGS,38.1,12340.0,343.4,1369.114735
15,2024-08-01 04:00:00,2024-07-31,4,CGS,38.2,12340.0,343.4,1372.708212
20,2024-08-01 05:00:00,2024-07-31,5,CGS,38.1,12340.0,343.4,1369.114735


In [33]:

merged_df["hourly_gas_burn"] = ((merged_df["daily_gas_burn"] / merged_df["daily_site_gen_mw"]) * merged_df["hourly_site_gen_mw"]) 

merged_df.head()
#merged_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\Check 7-23 (1).csv", index=False)


,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw,hourly_gas_burn
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1,12340.0,343.4,1369.114735
5,2024-08-01 02:00:00,2024-07-31,2,CGS,38.0,12340.0,343.4,1365.521258
10,2024-08-01 03:00:00,2024-07-31,3,CGS,38.1,12340.0,343.4,1369.114735
15,2024-08-01 04:00:00,2024-07-31,4,CGS,38.2,12340.0,343.4,1372.708212
20,2024-08-01 05:00:00,2024-07-31,5,CGS,38.1,12340.0,343.4,1369.114735


In [34]:
# final output of allegro data
result_df = merged_df[[ "datetime", "gas_day", "site", "hourly_site_gen_mw",  "daily_site_gen_mw", "daily_gas_burn", "hourly_gas_burn"]]


#math for daily MMBtu per MWh
result_df["gas_per_mw"] = (result_df["daily_gas_burn"] / result_df["daily_site_gen_mw"])

print(result_df.shape)
result_df.head(10)


(87600, 8)


,datetime,gas_day,site,hourly_site_gen_mw,daily_site_gen_mw,daily_gas_burn,hourly_gas_burn,gas_per_mw
0,2024-08-01 01:00:00,2024-07-31,CGS,38.1,343.4,12340.0,1369.114735,35.93477
5,2024-08-01 02:00:00,2024-07-31,CGS,38.0,343.4,12340.0,1365.521258,35.93477
10,2024-08-01 03:00:00,2024-07-31,CGS,38.1,343.4,12340.0,1369.114735,35.93477
15,2024-08-01 04:00:00,2024-07-31,CGS,38.2,343.4,12340.0,1372.708212,35.93477
20,2024-08-01 05:00:00,2024-07-31,CGS,38.1,343.4,12340.0,1369.114735,35.93477
25,2024-08-01 06:00:00,2024-07-31,CGS,38.3,343.4,12340.0,1376.301689,35.93477
30,2024-08-01 07:00:00,2024-07-31,CGS,38.0,343.4,12340.0,1365.521258,35.93477
35,2024-08-01 08:00:00,2024-07-31,CGS,38.1,343.4,12340.0,1369.114735,35.93477
40,2024-08-01 09:00:00,2024-07-31,CGS,38.5,343.4,12340.0,1383.488643,35.93477
45,2024-08-01 10:00:00,2024-08-01,CGS,38.6,1271.2,13565.0,411.901353,10.67102


In [35]:
export_df = result_df.copy()

#making a calendar date separate from gas day
export_df["date"] = export_df["datetime"].dt.date

export_df["gas_per_mw"] = (export_df["daily_gas_burn"] / export_df["daily_site_gen_mw"]).replace([np.inf, - np.inf], np.nan)

export_df = export_df[["datetime", "date", "gas_day", "site", "hourly_site_gen_mw", "daily_site_gen_mw", "daily_gas_burn", "hourly_gas_burn", "gas_per_mw"]]
export_df = export_df.sort_values(["site", "gas_day", "datetime"])


In [36]:
# checking number of records that would get excluded in the next cell
export_df[(export_df["daily_site_gen_mw"] > -1) & (export_df["gas_per_mw"] > 5) & (export_df["gas_per_mw"] < 20)].shape #67992... too many

(67992, 9)

In [37]:
# CALEB NOTE: Omitting data may not be the best approach at all especially if it is legit
#I was trying to clean values but I am not sure what the numbers/parameter should be 
#commenting out next line b/c it would remove ~68K records
#clean_df = export_df[(export_df["daily_site_gen_mw"] > -1) & (export_df["gas_per_mw"] > 5) & (export_df["gas_per_mw"] < 20)]

clean_df = export_df


export_df["gas_per_mw"] = pd.to_numeric(export_df["gas_per_mw"], errors="coerce")




In [38]:
print(merged_df.shape)
print(result_df.shape)

(87600, 8)
(87600, 8)


In [39]:
#Adding in an export to csv to output the data thus far
result_df = merged_df[["datetime", "gas_day", "site", "daily_site_gen_mw", "hourly_site_gen_mw", "daily_gas_burn", "hourly_gas_burn"]]

#result_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\allegro_data.csv", index=False) 



In [ ]:
#Importing the Unit Availability data from Excel and CSV files from the G drive 
excel_files = [
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 07.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 06.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 05.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 04.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 03.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 02.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 01.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 12.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 11.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 10.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 09.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 08.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 07.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 06.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 05.25 - UPDATE.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 04.25 - UPDATE.xlsx"]

csv_files = [
    r"G:\Trading\Market Operations\Unit availability\2025\dpm_BEPC_GROUPING_2025030100_2025033123 - March.csv",
    r"G:\Trading\Market Operations\Unit availability\2025\transposed_dpm_BEPC_GROUPING_2025020100_2025022823 feb.csv",
    r"G:\Trading\Market Operations\Unit availability\2025\transposed_dpm_BEPC_GROUPING_2025010100_2025013123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024120100_2024123123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024110100_2024113023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024100100_2024103123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024090100_2024093023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024080100_2024083123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024070100_2024073123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024060100_2024063023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024050100_2024053123.csv"]


In [41]:
def pull_unit_availability(excel_files, csv_files):


    excel_dfs = []
    for file in excel_files:
        df = pd.read_excel(file, sheet_name="Gas HEL Transposed")
        df["source_file"] = file
        excel_dfs.append(df)

    excel_combined = pd.concat(excel_dfs, ignore_index=True)

    csv_dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        df["source_file"] = file
        csv_dfs.append(df)

    csv_combined = pd.concat(csv_dfs, ignore_index=True)

    #Concat the csvs and the excel files
    combined_df = pd.concat([excel_combined, csv_combined], ignore_index=True)

    #Making all column names lowercase and remove spaces
    combined_df.columns = (combined_df.columns.str.strip().str.lower())

    #Making datetime column in datetime format
    combined_df["datetime"] = pd.to_datetime( combined_df["datetime"], errors="coerce")

    #Sorting chronologically
    combined_df = (combined_df.sort_values("datetime").reset_index(drop=True))

    return combined_df




In [42]:
#Load data from the excel and csv files 
df = pull_unit_availability(excel_files, csv_files)

In [43]:
#this is to defragment and get the performance warning to go away
df = df.copy()

# clean columns
df.columns = df.columns.astype(str) 
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)


In [44]:
# datetime fix
df = df.rename(columns={"DateTime": "datetime"})
df["datetime"] = pd.to_datetime(df["datetime"].astype(str), errors="coerce")

print(f'rows before dropping na datetimes: {df.shape}')
df = df.dropna(subset=["datetime"]) # 50 records
print(f'rows after dropping na datetimes: {df.shape}')


rows before dropping na datetimes: (19892, 35)
rows after dropping na datetimes: (19842, 35)


In [45]:
df.columns

Index(['datetime', 'cgs1 - high effective limit',
       'dcs1 - high effective limit', 'ggs1 - high effective limit',
       'ggs2 - high effective limit', 'lcs1 - high effective limit',
       'lcs2 - high effective limit', 'lcs3 - high effective limit',
       'lcs4 - high effective limit', 'lcs5 - high effective limit',
       'lcs6 - high effective limit', 'pgs1 - high effective limit',
       'pgs2 - high effective limit', 'pgs3 - high effective limit',
       'pgs11 - high effective limit', 'pgs12 - high effective limit',
       'pgs13 - high effective limit', 'pgs14 - high effective limit',
       'pgs15 - high effective limit', 'pgs16 - high effective limit',
       'pgs17 - high effective limit', 'pgs18 - high effective limit',
       'pgs19 - high effective limit', 'pgs20 - high effective limit',
       'pgs21 - high effective limit', 'pgs22 - high effective limit',
       'pgs31 - high effective limit', 'pgs32 - high effective limit',
       'pgs33 - high effective limit', 

In [46]:
#Finding all High Effective Limit availability columns 
value_cols = [col for col in df.columns if isinstance(col, str) and "high effective limit" in col]



In [47]:
# melt (wide data to long data)
availability = df.melt(id_vars=["datetime"], value_vars=value_cols, var_name="unit", value_name="availability_mw")

print(availability.shape)
availability.head


(654786, 3)


<bound method NDFrame.head of                   datetime                         unit  availability_mw
0      2024-05-01 00:00:00  cgs1 - high effective limit             45.0
1      2024-05-01 01:00:00  cgs1 - high effective limit             45.0
2      2024-05-01 02:00:00  cgs1 - high effective limit             45.0
3      2024-05-01 03:00:00  cgs1 - high effective limit             45.0
4      2024-05-01 04:00:00  cgs1 - high effective limit             45.0
...                    ...                          ...              ...
654781 2026-07-01 19:00:00  pgs5 - high effective limit            225.0
654782 2026-07-01 20:00:00  pgs5 - high effective limit            225.0
654783 2026-07-01 21:00:00  pgs5 - high effective limit            225.0
654784 2026-07-01 22:00:00  pgs5 - high effective limit            225.0
654785 2026-07-01 23:00:00  pgs5 - high effective limit            225.0

[654786 rows x 3 columns]>

In [ ]:
#Checkpoint #11
#df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint11.csv")

In [48]:
#Making sure the values are numeric so we can do the utilization calc
availability["availability_mw"] = pd.to_numeric(availability["availability_mw"], errors="coerce")


In [49]:
availability.head()

,datetime,unit,availability_mw
0,2024-05-01 00:00:00,cgs1 - high effective limit,45.0
1,2024-05-01 01:00:00,cgs1 - high effective limit,45.0
2,2024-05-01 02:00:00,cgs1 - high effective limit,45.0
3,2024-05-01 03:00:00,cgs1 - high effective limit,45.0
4,2024-05-01 04:00:00,cgs1 - high effective limit,45.0


In [50]:
# extract site from unit name
availability["site"] = availability["unit"].str.upper().str.extract(r"^(CGS|DCS|GGS|LCS|PGS)")
availability["site"] = availability["site"].str.strip().str.upper()


In [51]:
availability.head()

,datetime,unit,availability_mw,site
0,2024-05-01 00:00:00,cgs1 - high effective limit,45.0,CGS
1,2024-05-01 01:00:00,cgs1 - high effective limit,45.0,CGS
2,2024-05-01 02:00:00,cgs1 - high effective limit,45.0,CGS
3,2024-05-01 03:00:00,cgs1 - high effective limit,45.0,CGS
4,2024-05-01 04:00:00,cgs1 - high effective limit,45.0,CGS


In [52]:
# aggregate to site level 
site_availability = (availability.groupby(["datetime", "site"], as_index=False)["availability_mw"].sum())


In [53]:
site_availability.head()

,datetime,site,availability_mw
0,2024-05-01,CGS,45.0
1,2024-05-01,DCS,297.0
2,2024-05-01,GGS,0.0
3,2024-05-01,LCS,143.0
4,2024-05-01,PGS,131.8


In [54]:
site_availability.shape

(90940, 3)

In [55]:
site_availability['datetime'].max()

Timestamp('2026-07-01 23:00:00')

In [ ]:
#Checkpoint #12
#site_availability.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint12.csv")

In [56]:
# merge to the rest of the df so far
master_df = result_df.copy()

master_df = master_df.merge(site_availability, on=["datetime", "site"], how="left")

In [57]:
## Caleb Check - Almost no availability data at this time for July 2026
master_df[master_df['datetime'] >= '2026-07-01'].to_csv('./output-data/july data check.csv', index=False)

In [ ]:
#Checkpoint #13
#master_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint13.csv")

In [58]:
#not sure how much is repeated here but every time I remove a line it stops working
def load_unit_availability_by_site():
    
    df = pull_unit_availability(excel_files, csv_files)

    #Change column names to strings and strip whitespace
    df.columns = df.columns.map(lambda x: str(x).strip())

    df = df.rename(columns={"DateTime": "datetime"})

    #Standardize datetime 
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

    # Only keep HEL columns
    hel_cols = [c for c in df.columns if "high effective limit" in c.lower()]

    df[hel_cols] = df[hel_cols].apply(pd.to_numeric, errors="coerce")

    df_long = df.melt(id_vars="datetime", value_vars=hel_cols, var_name="unit", value_name="availability_mw")

    df_long["site"] = df_long["unit"].str.extract(r"^(cgs|dcs|ggs|lcs|pgs)", expand=False).str.upper()

    availability_by_site = (df_long.groupby(["datetime", "site"], as_index=False).agg({"availability_mw": "sum"}))

    return availability_by_site


In [59]:
site_df = load_unit_availability_by_site()

In [60]:
print(site_df.shape)
site_df.head()

(90940, 3)


,datetime,site,availability_mw
0,2024-05-01,CGS,45.0
1,2024-05-01,DCS,297.0
2,2024-05-01,GGS,0.0
3,2024-05-01,LCS,143.0
4,2024-05-01,PGS,131.8


In [61]:
print(site_df["datetime"].min())
print(site_df["datetime"].max())

2024-05-01 00:00:00
2026-07-01 23:00:00


In [62]:
#merge data to master df
master_df = master_df.merge(site_df, on=["datetime", "site"], how="left", suffixes=("", "_actual"))
master_df = master_df.drop_duplicates(["datetime", "site"])


In [63]:
print(master_df.shape)
master_df.head()

(87600, 9)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,availability_mw_actual
0,2024-08-01 01:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,40.0
1,2024-08-01 02:00:00,2024-07-31,CGS,343.4,38.0,12340.0,1365.521258,40.0,40.0
2,2024-08-01 03:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,40.0
3,2024-08-01 04:00:00,2024-07-31,CGS,343.4,38.2,12340.0,1372.708212,40.0,40.0
4,2024-08-01 05:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,40.0


In [64]:
print(master_df["datetime"].min())
print(master_df["datetime"].max())

2024-08-01 01:00:00
2026-08-01 00:00:00


In [65]:
master_df.head()

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,availability_mw_actual
0,2024-08-01 01:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,40.0
1,2024-08-01 02:00:00,2024-07-31,CGS,343.4,38.0,12340.0,1365.521258,40.0,40.0
2,2024-08-01 03:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,40.0
3,2024-08-01 04:00:00,2024-07-31,CGS,343.4,38.2,12340.0,1372.708212,40.0,40.0
4,2024-08-01 05:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,40.0


In [66]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   datetime                87600 non-null  datetime64[us]
 1   gas_day                 87600 non-null  datetime64[s] 
 2   site                    87600 non-null  str           
 3   daily_site_gen_mw       87600 non-null  float64       
 4   hourly_site_gen_mw      87600 non-null  float64       
 5   daily_gas_burn          86565 non-null  float64       
 6   hourly_gas_burn         69429 non-null  float64       
 7   availability_mw         80135 non-null  float64       
 8   availability_mw_actual  80135 non-null  float64       
dtypes: datetime64[s](1), datetime64[us](1), float64(6), str(1)
memory usage: 6.3 MB


In [67]:
#calculate utilization rate
master_df["utilization"] = (master_df["hourly_site_gen_mw"] / master_df["availability_mw"])

master_df["utilization"] = master_df["utilization"].replace([np.inf, -np.inf], np.nan)
master_df["utilization"].head()

0    0.9525
1    0.9500
2    0.9525
3    0.9550
4    0.9525
Name: utilization, dtype: float64

In [ ]:
#Checkpoint #14
#master_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint14.csv")

In [68]:
#to make sure the variables for the api are on the local os  
YES_USER = os.getenv("YES_USERNAME")
YES_PASS = os.getenv("YES_PASSWORD")

print("YES_USER loaded:", YES_USER is not None)
print("YES_PASS loaded:", YES_PASS is not None)


YES_USER loaded: True
YES_PASS loaded: True


In [69]:
def pull_yes_forecast_historical(user, password, start_date, end_date):

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    df_forecast = []

    while start <= end:

        month_start = start.replace(day=1)
        month_end = month_start + pd.offsets.MonthEnd(1)

        if month_end > end:
            month_end = end

        print(f"Pulling forecast: {month_start.date()} ---> {month_end.date()}")

        url = ( "https://services.yesenergy.com/PS/rest/timeseries/multiple.json?agglevel=hour&timezone=CPT"
            f"&startdate={month_start.date()}"
            f"&enddate={month_end.date()}"
            "&items="
            "LOAD_FORECAST:10017060648,"
            "NET_LOAD_FORECAST_CURRENT:10017060648,"
            "NG_CAPACITY_OFFLINE:10017060648,"
            "COAL_CAPACITY_OFFLINE:10017060648,"
            "WINDFCST_HOURLY:10004185377,"
            "WINDFCST_HOURLY:10004185378,"
            "WINDFCST_HOURLY:10004185379,"
            "WINDFCST_HOURLY:10004185380,"
            "WINDFCST_HOURLY:10004185381,"
            "WSI_FC15_FEEL:10000355230,"
            "WSI_FC15_FEEL:10000355704,"
            "WSI_FC15_FEEL:10000356081,"
            "WSI_FC15_WIND:10000355230,"
            "WSI_FC15_WIND:10000355704" )

        response = requests.get(url, auth=(user, password), verify=False, timeout=120)
        response.raise_for_status()
        
        # processing after successful data pull
        df_chunk = pd.DataFrame(response.json())
       
        df_chunk.columns = df_chunk.columns.map(lambda x: str(x).strip())

        df_forecast.append(df_chunk)

        time.sleep(6)  

        start = month_end + timedelta(days=1)

    return pd.concat(df_forecast, ignore_index=True)



In [70]:
print(pd.to_datetime(master_df["datetime"]).max().date())
print(pd.to_datetime(result_df["datetime"]).max().date())

2026-08-01
2026-08-01


In [71]:
# CALEB NOTE: kinda slow and annoying to wait for but it does work
#call the API
#changed the date to be based on master_df
df_forecast = pull_yes_forecast_historical( YES_USER, YES_PASS, start_date="2023-06-01", end_date = pd.to_datetime(master_df["datetime"]).max().date())


Pulling forecast: 2023-06-01 ---> 2023-06-30
Pulling forecast: 2023-07-01 ---> 2023-07-31
Pulling forecast: 2023-08-01 ---> 2023-08-31
Pulling forecast: 2023-09-01 ---> 2023-09-30
Pulling forecast: 2023-10-01 ---> 2023-10-31
Pulling forecast: 2023-11-01 ---> 2023-11-30
Pulling forecast: 2023-12-01 ---> 2023-12-31
Pulling forecast: 2024-01-01 ---> 2024-01-31
Pulling forecast: 2024-02-01 ---> 2024-02-29
Pulling forecast: 2024-03-01 ---> 2024-03-31
Pulling forecast: 2024-04-01 ---> 2024-04-30
Pulling forecast: 2024-05-01 ---> 2024-05-31
Pulling forecast: 2024-06-01 ---> 2024-06-30
Pulling forecast: 2024-07-01 ---> 2024-07-31
Pulling forecast: 2024-08-01 ---> 2024-08-31
Pulling forecast: 2024-09-01 ---> 2024-09-30
Pulling forecast: 2024-10-01 ---> 2024-10-31
Pulling forecast: 2024-11-01 ---> 2024-11-30
Pulling forecast: 2024-12-01 ---> 2024-12-31
Pulling forecast: 2025-01-01 ---> 2025-01-31
Pulling forecast: 2025-02-01 ---> 2025-02-28
Pulling forecast: 2025-03-01 ---> 2025-03-31
Pulling fo

In [ ]:
#Checkpoint #13
#df_forecast.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint13.csv")

In [72]:
## CALEB NOTE: Be sure to check that the coercion isn't making improper conversions

def clean_yes_forecast(df):

    df = df_forecast.copy()

    #find datetime column
    datetime_col = [c for c in df.columns if "DATETIME" in c.upper()][0]

    df["datetime"] = pd.to_datetime(df[datetime_col], format="%m/%d/%Y %H:%M:%S", errors="coerce")

    #load 
    df["load"] = pd.to_numeric( df["SPPISO-East (LOAD_FORECAST)"], errors="coerce")

    #net load 
    df["net_load"] = pd.to_numeric(df["SPPISO-East (NET_LOAD_FORECAST_CURRENT)"], errors="coerce")

    #wind 
    wind_cols = [c for c in df.columns if "WINDFCST_HOURLY" in c]
    df["wind"] = df[wind_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)

    #outages 
    df["outage_ng"] = pd.to_numeric(df["SPPISO-East (NG_CAPACITY_OFFLINE)"], errors="coerce")

    df["outage_coal"] = pd.to_numeric(df["SPPISO-East (COAL_CAPACITY_OFFLINE)"], errors="coerce")

    #temperature avg of the zones (ask about this)
    temp_cols = [c for c in df.columns if "WSI_FC15_FEEL" in c]
    df["temperature"] = df[temp_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

    #wind speed avg of the reserve zones (also ask)
    wind_speed_cols = [c for c in df.columns if "WSI_FC15_WIND" in c]
    df["wind_speed"] = df[wind_speed_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

    #outage
    df["total_outages"] = df["outage_ng"] + df["outage_coal"]

    out = df[["datetime", "load", "net_load", "wind", "temperature", "wind_speed", "total_outages"]].dropna(subset=["datetime"])

    return out.sort_values("datetime").reset_index(drop=True)


In [73]:
#save the changes and get rid of forecast after for formatting
df_forecast = clean_yes_forecast(df_forecast)

In [74]:
forecast_columns = [columns for columns in master_df.columns if columns.endswith("_forecast")]
print(forecast_columns) #none exist
print(df_forecast.shape)

[]
(27792, 7)


In [75]:
print(df_forecast['datetime'].min())
print(df_forecast['datetime'].max())
df_forecast.head()


2023-06-01 01:00:00
2026-08-02 00:00:00


,datetime,load,net_load,wind,temperature,wind_speed,total_outages
0,2023-06-01 01:00:00,28065,11285.63,16779.37,68.480,5.90,12204.1
1,2023-06-01 02:00:00,27109,11246.52,15862.48,67.045,6.20,12204.1
2,2023-06-01 03:00:00,26469,10391.44,16077.56,65.750,5.90,12204.1
3,2023-06-01 04:00:00,25911,9448.65,16462.35,64.765,5.30,12764.1
4,2023-06-01 05:00:00,25752,11442.85,14309.15,60.070,7.75,13040.3


In [76]:
master_df = master_df.drop(columns=forecast_columns, errors="ignore")
print(df_forecast['datetime'].min())
print(df_forecast['datetime'].max())
df_forecast.head()

2023-06-01 01:00:00
2026-08-02 00:00:00


,datetime,load,net_load,wind,temperature,wind_speed,total_outages
0,2023-06-01 01:00:00,28065,11285.63,16779.37,68.480,5.90,12204.1
1,2023-06-01 02:00:00,27109,11246.52,15862.48,67.045,6.20,12204.1
2,2023-06-01 03:00:00,26469,10391.44,16077.56,65.750,5.90,12204.1
3,2023-06-01 04:00:00,25911,9448.65,16462.35,64.765,5.30,12764.1
4,2023-06-01 05:00:00,25752,11442.85,14309.15,60.070,7.75,13040.3


In [77]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   datetime                87600 non-null  datetime64[us]
 1   gas_day                 87600 non-null  datetime64[s] 
 2   site                    87600 non-null  str           
 3   daily_site_gen_mw       87600 non-null  float64       
 4   hourly_site_gen_mw      87600 non-null  float64       
 5   daily_gas_burn          86565 non-null  float64       
 6   hourly_gas_burn         69429 non-null  float64       
 7   availability_mw         80135 non-null  float64       
 8   availability_mw_actual  80135 non-null  float64       
 9   utilization             72697 non-null  float64       
dtypes: datetime64[s](1), datetime64[us](1), float64(7), str(1)
memory usage: 6.9 MB


In [78]:
df_forecast[df_forecast.duplicated(subset=['datetime'], keep=False)]

,datetime,load,net_load,wind,temperature,wind_speed,total_outages
3769,2023-11-05 02:00:00,25277,15988.94,9288.06,29.166667,4.500,NaN
3770,2023-11-05 02:00:00,24930,15707.29,9222.71,29.166667,4.500,22914.20
12505,2024-11-03 02:00:00,25795,12631.94,13163.06,40.270000,12.600,NaN
12506,2024-11-03 02:00:00,25385,12649.04,12735.96,40.270000,12.600,22870.94
21241,2025-11-02 02:00:00,28199,12430.19,15768.81,35.893333,13.675,NaN
21242,2025-11-02 02:00:00,28190,10970.57,17219.43,35.893333,13.675,18814.90


In [79]:
df_forecast.shape

(27792, 7)

In [80]:
df_forecast = df_forecast.drop_duplicates(subset=['datetime'], keep='last')

In [81]:
df_forecast.shape

(27789, 7)

In [82]:
#merge data to master df
master_df = master_df.merge(df_forecast, on="datetime", how="left", suffixes=("", "_forecast"))
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   datetime                87600 non-null  datetime64[us]
 1   gas_day                 87600 non-null  datetime64[s] 
 2   site                    87600 non-null  str           
 3   daily_site_gen_mw       87600 non-null  float64       
 4   hourly_site_gen_mw      87600 non-null  float64       
 5   daily_gas_burn          86565 non-null  float64       
 6   hourly_gas_burn         69429 non-null  float64       
 7   availability_mw         80135 non-null  float64       
 8   availability_mw_actual  80135 non-null  float64       
 9   utilization             72697 non-null  float64       
 10  load                    87590 non-null  float64       
 11  net_load                87590 non-null  float64       
 12  wind                    87590 non-null  float64       
 1

In [ ]:
# The join introduced a few duplicate rows because the yes energy data has multiple records for daylight savings hours in November.
# The following code will keep just the second instance of duplicated site/date combos
# Thiswill likely need to be evaluated further

#master_df = master_df.drop_duplicates(subset=['site', 'datetime'], keep='last')

In [83]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   datetime                87600 non-null  datetime64[us]
 1   gas_day                 87600 non-null  datetime64[s] 
 2   site                    87600 non-null  str           
 3   daily_site_gen_mw       87600 non-null  float64       
 4   hourly_site_gen_mw      87600 non-null  float64       
 5   daily_gas_burn          86565 non-null  float64       
 6   hourly_gas_burn         69429 non-null  float64       
 7   availability_mw         80135 non-null  float64       
 8   availability_mw_actual  80135 non-null  float64       
 9   utilization             72697 non-null  float64       
 10  load                    87590 non-null  float64       
 11  net_load                87590 non-null  float64       
 12  wind                    87590 non-null  float64       
 1

In [84]:

def pull_yes_actual_historical(user, password, start_date, end_date):

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    df_actual = []

    while start <= end:

        month_start = start.replace(day=1)
        month_end = month_start + pd.offsets.MonthEnd(1)

        if month_end > end:
            month_end = end

        print(f"Pulling actuals: {month_start.date()} ---> {month_end.date()}")

        url = ("https://services.yesenergy.com/PS/rest/timeseries/multiple.json?agglevel=hour&timezone=CPT"
            f"&startdate={month_start.date()}"
            f"&enddate={month_end.date()}"
            "&items="
            #day ahead close so just in actual
            "BIDCLOSE_LOAD_FORECAST:10017060648,"
            "NET_LOAD_FORECAST_BID_CLOSE:10017060648,"
            "NG_CAPACITY_OFFLINE:10017060648,"
            "COAL_CAPACITY_OFFLINE:10017060648,"
            "WINDGEN_HOURLY:10004185377,"
            "WINDGEN_HOURLY:10004185378,"
            "WINDGEN_HOURLY:10004185379,"
            "WINDGEN_HOURLY:10004185380,"
            "WINDGEN_HOURLY:10004185381,"
            "WSI_TRADER_FEELS_TEMP:10000355230,"
            "WSI_TRADER_FEELS_TEMP:10000355704,"
            "WSI_TRADER_FEELS_TEMP:10000356081,"
            "WSI_TRADER_WIND:10000355230,"
            "WSI_TRADER_WIND:10000355704")


        response = requests.get(url, auth=(user, password), verify=False, timeout=120)
        #response.raise_for_status()


        df_chunk = pd.DataFrame(response.json())
        df_chunk.columns = df_chunk.columns.map(lambda x: str(x).strip())

        df_actual.append(df_chunk)
        
        time.sleep(6)  

        start = month_end + timedelta(days=1)

    return pd.concat(df_actual, ignore_index=True)



In [85]:
def clean_yes_actual(df):
  
    # this doesn't actually copy anything
    df = df_actual.copy()

    datetime_col = [c for c in df.columns if "DATETIME" in c.upper()][0]

    df["datetime"] = pd.to_datetime( df[datetime_col], errors="coerce")


    # load actual
    df["load_actual"] = pd.to_numeric(df["SPPISO-East (BIDCLOSE_LOAD_FORECAST)"], errors="coerce")

    # net load actual
    df["net_load_actual"] = pd.to_numeric( df["SPPISO-East (NET_LOAD_FORECAST_BID_CLOSE)"], errors="coerce")

    # wind actual
    wind_cols = [c for c in df.columns if "WINDGEN_HOURLY" in c]
    wind_numeric = (df[wind_cols].apply(pd.to_numeric, errors="coerce"))

    wind_sum = wind_numeric.sum(axis=1, min_count=1)

    all_zero = (wind_numeric.fillna(0).sum(axis=1).eq(0))

    df["wind_actual"] = wind_sum.mask(all_zero)

    # outage actual
    df["outage_ng"] = pd.to_numeric(df["SPPISO-East (NG_CAPACITY_OFFLINE)"], errors="coerce")
    df["outage_coal"] = pd.to_numeric( df["SPPISO-East (COAL_CAPACITY_OFFLINE)"], errors="coerce")

    # temperature actual
    temp_cols = [c for c in df.columns if "WSI_TRADER_FEELS_TEMP" in c]
    temp_numeric = (df[temp_cols].apply(pd.to_numeric, errors="coerce"))

    temp_avg = temp_numeric.mean(axis=1)

    all_zero_temp = (temp_numeric.fillna(0).sum(axis=1).eq(0))

    df["temperature_actual"] = temp_avg.mask(all_zero_temp)

    # windspeed actual
    wind_speed_cols = [c for c in df.columns if "WSI_TRADER_WIND" in c]
    df["wind_speed_actual"] = df[wind_speed_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    
    
    # outage actual
    df["total_outages"] = df["outage_ng"] + df["outage_coal"]

    out = df[["datetime", "load_actual", "net_load_actual", "wind_actual", "temperature_actual", "wind_speed_actual", "total_outages"]].dropna(subset=["datetime"])

    return out.sort_values("datetime").reset_index(drop=True)


In [86]:

#call the API
# changed to pull the end date from master_df
df_actual = pull_yes_actual_historical( YES_USER, YES_PASS, start_date="2023-06-01", end_date = pd.to_datetime(master_df["datetime"]).max().date())


Pulling actuals: 2023-06-01 ---> 2023-06-30
Pulling actuals: 2023-07-01 ---> 2023-07-31
Pulling actuals: 2023-08-01 ---> 2023-08-31
Pulling actuals: 2023-09-01 ---> 2023-09-30
Pulling actuals: 2023-10-01 ---> 2023-10-31
Pulling actuals: 2023-11-01 ---> 2023-11-30
Pulling actuals: 2023-12-01 ---> 2023-12-31
Pulling actuals: 2024-01-01 ---> 2024-01-31
Pulling actuals: 2024-02-01 ---> 2024-02-29
Pulling actuals: 2024-03-01 ---> 2024-03-31
Pulling actuals: 2024-04-01 ---> 2024-04-30
Pulling actuals: 2024-05-01 ---> 2024-05-31
Pulling actuals: 2024-06-01 ---> 2024-06-30
Pulling actuals: 2024-07-01 ---> 2024-07-31
Pulling actuals: 2024-08-01 ---> 2024-08-31
Pulling actuals: 2024-09-01 ---> 2024-09-30
Pulling actuals: 2024-10-01 ---> 2024-10-31
Pulling actuals: 2024-11-01 ---> 2024-11-30
Pulling actuals: 2024-12-01 ---> 2024-12-31
Pulling actuals: 2025-01-01 ---> 2025-01-31
Pulling actuals: 2025-02-01 ---> 2025-02-28
Pulling actuals: 2025-03-01 ---> 2025-03-31
Pulling actuals: 2025-04-01 --->

In [87]:
df_actual.info()

<class 'pandas.DataFrame'>
RangeIndex: 27792 entries, 0 to 27791
Data columns (total 20 columns):
 #   Column                                           Non-Null Count  Dtype 
---  ------                                           --------------  ----- 
 0   DATETIME                                         27792 non-null  str   
 1   SPPISO-East (BIDCLOSE_LOAD_FORECAST)             27792 non-null  str   
 2   SPPISO-East (NET_LOAD_FORECAST_BID_CLOSE)        27792 non-null  str   
 3   SPPISO-East (NG_CAPACITY_OFFLINE)                27786 non-null  str   
 4   SPPISO-East (COAL_CAPACITY_OFFLINE)              27786 non-null  str   
 5   RESERVE ZONE 1 (WINDGEN_HOURLY)                  27777 non-null  str   
 6   RESERVE ZONE 2 (WINDGEN_HOURLY)                  27777 non-null  str   
 7   RESERVE ZONE 3 (WINDGEN_HOURLY)                  27777 non-null  str   
 8   RESERVE ZONE 4 (WINDGEN_HOURLY)                  27777 non-null  str   
 9   RESERVE ZONE 5 (WINDGEN_HOURLY)                  2

In [179]:
#now cleaned
df_actual_clean = clean_yes_actual(df_actual)


In [180]:
#making sure if YES changes how they format datetime the code still runs
datetime_column = [columns for columns in df_actual.columns
    if "DATETIME" in columns.upper()][0]

columns_to_drop = [c for c in master_df.columns if c.endswith("_actual")]

master_df = master_df.drop(columns=columns_to_drop, errors="ignore")


In [181]:
master_df.head()

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,gas_per_mw,load_final,wind_final,temperature_final,hour,day_of_week,month,gas_lag_1,gas_lag_24,gas_roll_24
0,2024-08-01 01:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,39455.0,...,35.93477,40690.0,16094.141,68.666667,1,3,8,NaN,NaN,NaN
1,2024-08-01 02:00:00,2024-07-31,CGS,343.4,38.0,12340.0,1365.521258,40.0,0.9500,37842.0,...,35.93477,39016.0,15461.807,67.666667,2,3,8,1369.114735,NaN,NaN
2,2024-08-01 03:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,35806.0,...,35.93477,37657.0,14812.445,66.333333,3,3,8,1365.521258,NaN,NaN
3,2024-08-01 04:00:00,2024-07-31,CGS,343.4,38.2,12340.0,1372.708212,40.0,0.9550,34889.0,...,35.93477,36631.0,13490.468,67.000000,4,3,8,1369.114735,NaN,NaN
4,2024-08-01 05:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,34518.0,...,35.93477,36153.0,12879.901,65.666667,5,3,8,1372.708212,NaN,NaN


In [ ]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            87600 non-null  datetime64[us]
 1   gas_day             87600 non-null  datetime64[s] 
 2   site                87600 non-null  str           
 3   daily_site_gen_mw   87600 non-null  float64       
 4   hourly_site_gen_mw  87600 non-null  float64       
 5   daily_gas_burn      86565 non-null  float64       
 6   hourly_gas_burn     69429 non-null  float64       
 7   availability_mw     80135 non-null  float64       
 8   utilization         72697 non-null  float64       
 9   load                87590 non-null  float64       
 10  net_load            87590 non-null  float64       
 11  wind                87590 non-null  float64       
 12  temperature         87590 non-null  float64       
 13  wind_speed          87590 non-null  float64       
 14  t

In [ ]:
df_actual_clean[df_actual_clean.duplicated(subset=['datetime'], keep=False)]

print(f'rows before dropping duplicates: {df_actual_clean.shape}')
df_actual_clean = df_actual_clean.drop_duplicates(subset=['datetime'], keep='last')
print(f'rows after dropping duplicates: {df_actual_clean.shape}')

rows before dropping duplicates: (27792, 7)
rows after dropping duplicates: (27789, 7)


In [93]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            87600 non-null  datetime64[us]
 1   gas_day             87600 non-null  datetime64[s] 
 2   site                87600 non-null  str           
 3   daily_site_gen_mw   87600 non-null  float64       
 4   hourly_site_gen_mw  87600 non-null  float64       
 5   daily_gas_burn      86565 non-null  float64       
 6   hourly_gas_burn     69429 non-null  float64       
 7   availability_mw     80135 non-null  float64       
 8   utilization         72697 non-null  float64       
 9   load                87590 non-null  float64       
 10  net_load            87590 non-null  float64       
 11  wind                87590 non-null  float64       
 12  temperature         87590 non-null  float64       
 13  wind_speed          87590 non-null  float64       
 14  t

In [94]:
#merge data to master df
master_df = master_df.merge(df_actual_clean, on="datetime", how="left", suffixes=("", "_actual"))
master_df = master_df.drop_duplicates(["datetime", "site"])


In [95]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              87600 non-null  datetime64[us]
 1   gas_day               87600 non-null  datetime64[s] 
 2   site                  87600 non-null  str           
 3   daily_site_gen_mw     87600 non-null  float64       
 4   hourly_site_gen_mw    87600 non-null  float64       
 5   daily_gas_burn        86565 non-null  float64       
 6   hourly_gas_burn       69429 non-null  float64       
 7   availability_mw       80135 non-null  float64       
 8   utilization           72697 non-null  float64       
 9   load                  87590 non-null  float64       
 10  net_load              87590 non-null  float64       
 11  wind                  87590 non-null  float64       
 12  temperature           87590 non-null  float64       
 13  wind_speed            87590

In [96]:
master_df.head()

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,wind,temperature,wind_speed,total_outages,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages_actual
0,2024-08-01 01:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,39455.0,...,18134.43,69.62,5.25,6533.16,40690.0,21511.77,16094.141,68.666667,2.30,6533.16
1,2024-08-01 02:00:00,2024-07-31,CGS,343.4,38.0,12340.0,1365.521258,40.0,0.9500,37842.0,...,16886.50,68.18,5.30,6533.16,39016.0,19919.28,15461.807,67.666667,2.85,6533.16
2,2024-08-01 03:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,35806.0,...,15508.85,67.04,5.30,6533.16,37657.0,18746.28,14812.445,66.333333,3.45,6533.16
3,2024-08-01 04:00:00,2024-07-31,CGS,343.4,38.2,12340.0,1372.708212,40.0,0.9550,34889.0,...,14530.44,65.90,4.35,6533.16,36631.0,18405.92,13490.468,67.000000,2.85,6533.16
4,2024-08-01 05:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,34518.0,...,13977.19,66.37,2.80,6858.16,36153.0,18817.29,12879.901,65.666667,4.00,6858.16


In [97]:
master_df.to_csv('./output-data/master_df modelling check.csv', index=False)

### **Caleb started making exstensive changes starting here**
Changes include splitting into multiple cells and adding checks

In [98]:
#Caleb Check
print(master_df.shape)
master_df.head()

(87600, 21)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,wind,temperature,wind_speed,total_outages,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages_actual
0,2024-08-01 01:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,39455.0,...,18134.43,69.62,5.25,6533.16,40690.0,21511.77,16094.141,68.666667,2.30,6533.16
1,2024-08-01 02:00:00,2024-07-31,CGS,343.4,38.0,12340.0,1365.521258,40.0,0.9500,37842.0,...,16886.50,68.18,5.30,6533.16,39016.0,19919.28,15461.807,67.666667,2.85,6533.16
2,2024-08-01 03:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,35806.0,...,15508.85,67.04,5.30,6533.16,37657.0,18746.28,14812.445,66.333333,3.45,6533.16
3,2024-08-01 04:00:00,2024-07-31,CGS,343.4,38.2,12340.0,1372.708212,40.0,0.9550,34889.0,...,14530.44,65.90,4.35,6533.16,36631.0,18405.92,13490.468,67.000000,2.85,6533.16
4,2024-08-01 05:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,34518.0,...,13977.19,66.37,2.80,6858.16,36153.0,18817.29,12879.901,65.666667,4.00,6858.16


In [99]:
#master_df["gas_per_mw"] = ( master_df["hourly_gas_burn"] / master_df["hourly_mw"])
master_df["gas_per_mw"] = ( master_df["hourly_gas_burn"] / master_df["hourly_site_gen_mw"]) # Caleb Adjusted Column Reference

master_df["gas_per_mw"] = master_df["gas_per_mw"].replace([np.inf, -np.inf], np.nan)


In [100]:
#Caleb Check
master_df.head()
print(master_df.shape)
master_df['gas_per_mw'].isna().sum()

(87600, 22)


np.int64(30805)

In [101]:
# Caleb Note so he remembers: filling blank actuals with the forecasted value is blank. Would interpolating from previous values be better? Might not actually matter since the number being replaced in relatively small
master_df["load_final"] = master_df["load_actual"].fillna(master_df["load"])
master_df["wind_final"] = master_df["wind_actual"].fillna(master_df["wind"])
master_df["temperature_final"] = master_df["temperature_actual"].fillna(master_df["temperature"])


In [103]:
# Caleb Check on number of null values. If large number likely b/c I changed the start date for YES Energy data pulls
print(f'NAs in actual load: {master_df["load_actual"].isna().sum()}')
print(f'NAs in actual load: {master_df["wind_actual"].isna().sum()}')
print(f'NAs in actual load: {master_df["temperature_actual"].isna().sum()}')

NAs in actual load: 10
NAs in actual load: 65
NAs in actual load: 140


In [104]:
#clean and prep
master_df["hourly_gas_burn"] = pd.to_numeric(master_df["hourly_gas_burn"], errors="coerce")


In [105]:
master_df[master_df['hourly_gas_burn']==0] #12641 rows

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages_actual,gas_per_mw,load_final,wind_final,temperature_final
90,2024-08-04 19:00:00,2024-08-04,CGS,438.1,0.0,4294.0,0.0,0.0,NaN,49432.0,...,48434.0,40237.306667,5571.642,72.000000,13.25,6297.16,NaN,48434.0,5571.642,72.000000
91,2024-08-04 20:00:00,2024-08-04,CGS,438.1,0.0,4294.0,0.0,0.0,NaN,48223.0,...,46909.0,37314.511667,6787.627,70.333333,10.90,6297.16,NaN,46909.0,6787.627,70.333333
92,2024-08-04 21:00:00,2024-08-04,CGS,438.1,0.0,4294.0,0.0,0.0,NaN,46408.0,...,45187.0,33300.036667,9325.150,67.666667,9.25,6428.16,NaN,45187.0,9325.150,67.666667
93,2024-08-04 22:00:00,2024-08-04,CGS,438.1,0.0,4294.0,0.0,0.0,NaN,44631.0,...,43585.0,28759.922500,13120.568,65.666667,8.05,6428.16,NaN,43585.0,13120.568,65.666667
94,2024-08-04 23:00:00,2024-08-04,CGS,438.1,0.0,4294.0,0.0,0.0,NaN,42081.0,...,41347.0,24574.109167,16901.844,65.000000,10.95,6428.16,NaN,41347.0,16901.844,65.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86767,2026-06-27 08:00:00,2026-06-26,PGS,4433.8,0.0,43717.0,0.0,768.3,0.0,31986.0,...,32493.0,11586.814167,16450.474,66.666667,11.50,5910.13,NaN,32493.0,16450.474,66.666667
86768,2026-06-27 09:00:00,2026-06-26,PGS,4433.8,0.0,43717.0,0.0,768.3,0.0,33708.0,...,34175.0,12642.274167,16664.841,67.666667,12.10,5910.13,NaN,34175.0,16664.841,67.666667
86769,2026-06-27 10:00:00,2026-06-27,PGS,1649.9,0.0,16131.0,0.0,768.3,0.0,35586.0,...,36036.0,13636.409167,17749.018,67.000000,14.40,7138.13,NaN,36036.0,17749.018,67.000000
86770,2026-06-27 11:00:00,2026-06-27,PGS,1649.9,0.0,16131.0,0.0,768.3,0.0,37304.0,...,37953.0,14855.907500,19279.991,68.000000,18.40,7138.13,NaN,37953.0,19279.991,68.000000


In [ ]:
# Caleb Note: I'm not sure that 0s should be replaced to nans especially if the 0 is legit
# remove fake zeros (look into this)
master_df.loc[ master_df["hourly_gas_burn"] == 0, "hourly_gas_burn"] = np.nan

In [106]:
master_df = master_df.sort_values(["site", "datetime"])

In [ ]:
#adding time features
master_df["hour"] = master_df["datetime"].dt.hour
master_df["day_of_week"] = master_df["datetime"].dt.dayofweek
master_df["month"] = master_df["datetime"].dt.month
master_df['year'] = master_df['datetime'].dt.year


#adding time features
master_df["gas_hour"] = master_df["gas_day"].dt.hour
master_df["gas_day_of_week"] = master_df["gas_day"].dt.dayofweek
master_df["gas_month"] = master_df["gas_day"].dt.month
master_df['gas_year'] = master_df['gas_day'].dt.year

In [ ]:
#adding lag features
# Caleb Note: these lags need to be more thought out. How do we handle blanks? Should we calculate averages when there are less than 24 records? Are blanks the equivalent of zeros in some of these cases
master_df["gas_lag_1"] = master_df.groupby("site")["hourly_gas_burn"].shift(1)

master_df["gas_lag_24"] = master_df.groupby("site")["hourly_gas_burn"].shift(23)# CALEB NOTE: I think this needs to be 23 not 24. Right now, a 1:00 AM record is pulling from the 2:00 AM record of the previous day

master_df["gas_roll_24"] = master_df.groupby("site")["hourly_gas_burn"].transform(lambda x: x.shift(1).rolling(24).mean())


In [109]:
# Caleb Check
master_df.to_csv('./output-data/lag features check.csv', index=False)
master_df.head()

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,gas_per_mw,load_final,wind_final,temperature_final,hour,day_of_week,month,gas_lag_1,gas_lag_24,gas_roll_24
0,2024-08-01 01:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,39455.0,...,35.93477,40690.0,16094.141,68.666667,1,3,8,NaN,NaN,NaN
1,2024-08-01 02:00:00,2024-07-31,CGS,343.4,38.0,12340.0,1365.521258,40.0,0.9500,37842.0,...,35.93477,39016.0,15461.807,67.666667,2,3,8,1369.114735,NaN,NaN
2,2024-08-01 03:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,35806.0,...,35.93477,37657.0,14812.445,66.333333,3,3,8,1365.521258,NaN,NaN
3,2024-08-01 04:00:00,2024-07-31,CGS,343.4,38.2,12340.0,1372.708212,40.0,0.9550,34889.0,...,35.93477,36631.0,13490.468,67.000000,4,3,8,1369.114735,NaN,NaN
4,2024-08-01 05:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,34518.0,...,35.93477,36153.0,12879.901,65.666667,5,3,8,1372.708212,NaN,NaN


In [110]:
## Caleb Check: how many would get dropped?
master_df.loc[:, ["hourly_gas_burn", "gas_lag_1", "gas_lag_24", "gas_roll_24"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   hourly_gas_burn  69429 non-null  float64
 1   gas_lag_1        69429 non-null  float64
 2   gas_lag_24       69429 non-null  float64
 3   gas_roll_24      65864 non-null  float64
dtypes: float64(4)
memory usage: 2.7 MB


In [111]:
## data set for model training

model_df = master_df.copy()

# commenting out next line b/c it would remove ~27K records
#model_df = model_df.dropna(subset=["hourly_gas_burn", "gas_lag_1", "gas_lag_24", "gas_roll_24"])

#model_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Training Datasets\Model Input Data YES Forecast Added and shift update.csv", index=False)

print(model_df.shape)


(87600, 31)


In [112]:
# Caleb Check
model_df.head()

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,gas_per_mw,load_final,wind_final,temperature_final,hour,day_of_week,month,gas_lag_1,gas_lag_24,gas_roll_24
0,2024-08-01 01:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,39455.0,...,35.93477,40690.0,16094.141,68.666667,1,3,8,NaN,NaN,NaN
1,2024-08-01 02:00:00,2024-07-31,CGS,343.4,38.0,12340.0,1365.521258,40.0,0.9500,37842.0,...,35.93477,39016.0,15461.807,67.666667,2,3,8,1369.114735,NaN,NaN
2,2024-08-01 03:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,35806.0,...,35.93477,37657.0,14812.445,66.333333,3,3,8,1365.521258,NaN,NaN
3,2024-08-01 04:00:00,2024-07-31,CGS,343.4,38.2,12340.0,1372.708212,40.0,0.9550,34889.0,...,35.93477,36631.0,13490.468,67.000000,4,3,8,1369.114735,NaN,NaN
4,2024-08-01 05:00:00,2024-07-31,CGS,343.4,38.1,12340.0,1369.114735,40.0,0.9525,34518.0,...,35.93477,36153.0,12879.901,65.666667,5,3,8,1372.708212,NaN,NaN


In [113]:
print(f'hourly gas burn NANs: {model_df['hourly_gas_burn'].isna().sum()}')
print(f'gas lag1 NANs: {model_df['gas_lag_1'].isna().sum()}')
print(f'gas lag24 NANs: {model_df['gas_lag_24'].isna().sum()}')
print(f'gas roll24 NANs: {model_df['gas_roll_24'].isna().sum()}')

hourly gas burn NANs: 18171
gas lag1 NANs: 18171
gas lag24 NANs: 18171
gas roll24 NANs: 21736


In [114]:
#final dataset
model_df = master_df.copy()


## CALEB NOTE: I'm not sure we shoud be dropping NAs here/ Commenting out b/c this will cause incomplete records
#model_df = model_df.dropna(subset=[ "hourly_gas_burn", "gas_lag_1", "gas_lag_24", "gas_roll_24"])
model_df = model_df.sort_values(["site", "datetime"])


model_export = model_df.copy()

model_export = model_export.sort_values(  ["site", "datetime"])

#model_export.to_csv( r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\MASTER_MERGED_DATA.csv", index=False)


In [115]:
print("Exported MODEL_INPUT_DATA.csv")
print("Rows:", len(model_export))
print("Columns:", len(model_export.columns))


print("\n===== FINAL MODEL SUMMARY =====")

print(model_df.groupby("site").size())

print("\nMissing Values")
print(model_df[[ "hourly_gas_burn", "availability_mw", "utilization", "gas_lag_1", "gas_lag_24", "gas_roll_24" ]].isna().sum())

print("\nGas Per MW")
print(model_df.groupby("site")["gas_per_mw"].describe())

Exported MODEL_INPUT_DATA.csv
Rows: 87600
Columns: 31

===== FINAL MODEL SUMMARY =====
site
CGS    17520
DCS    17520
GGS    17520
LCS    17520
PGS    17520
dtype: int64

Missing Values
hourly_gas_burn    18171
availability_mw     7465
utilization        14903
gas_lag_1          18171
gas_lag_24         18171
gas_roll_24        21736
dtype: int64

Gas Per MW
        count       mean       std       min       25%        50%        75%  \
site                                                                           
CGS    6779.0  10.038447  1.670745  0.000000  9.621776  10.025888  10.187984   
DCS   11272.0   7.962162  2.592986  2.219512  7.478750   7.576692   7.772819   
GGS    6068.0   9.716082  4.149318  0.000000  9.107277   9.548313   9.924528   
LCS   15769.0  10.035412  2.882739  6.028993  9.786805   9.926977  10.084453   
PGS   16907.0   9.496828  1.320489  3.311697  9.487062   9.779168  10.033899   

             max  
site              
CGS    76.351351  
DCS   100.000000  
GG

In [116]:
# Caleb Outputting modelling dataframe to limit the amount of reloading needed
model_df.to_csv('./output-data/model_df_export.csv', index=False)

In [ ]:
# Caleb NOTE: This imputation may need to be adjusted to a better method in the future. Median might not make sense in all situations
# Additionally imputation should be done on the split dataset. Right now, this will introduce data leakage from the test and validation set
model_df["availability_mw"] = model_df.groupby("site")["availability_mw"].transform( lambda x: x.fillna(x.median()))


#listing the features
features = ["load_final", "wind_final", "temperature_final", "availability_mw", "utilization", "hour", "day_of_week", "month", "gas_lag_1", "gas_lag_24", "gas_roll_24"]

In [118]:
# Caleb Checks
model_df['site'].unique()

<ArrowStringArray>
['CGS', 'DCS', 'GGS', 'LCS', 'PGS']
Length: 5, dtype: str

In [119]:
## Caleb Testing
model_df[model_df['site']=='DCS']['datetime'].to_csv('./output-data/DCS Date Checks.csv', index=False)

In [120]:
## Caleb getting info for current model_df
print(model_df.shape)
model_df.info()

(87600, 31)
<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              87600 non-null  datetime64[us]
 1   gas_day               87600 non-null  datetime64[s] 
 2   site                  87600 non-null  str           
 3   daily_site_gen_mw     87600 non-null  float64       
 4   hourly_site_gen_mw    87600 non-null  float64       
 5   daily_gas_burn        86565 non-null  float64       
 6   hourly_gas_burn       69429 non-null  float64       
 7   availability_mw       87600 non-null  float64       
 8   utilization           72697 non-null  float64       
 9   load                  87590 non-null  float64       
 10  net_load              87590 non-null  float64       
 11  wind                  87590 non-null  float64       
 12  temperature           87590 non-null  float64       
 13  wind_speed     

In [121]:
### Caleb outputting model data so that I don't have to reload from scratch if I have to restart. I don't want to reload the YES Energy Data all the time
model_df.to_excel('./output-data/model_df.xlsx', index=False)

In [3]:
## Caleb importing model df if restart is needed
model_df = pd.read_excel('./output-data/model_df.xlsx')
model_df.columns

Index(['datetime', 'gas_day', 'site', 'daily_site_gen_mw',
       'hourly_site_gen_mw', 'daily_gas_burn', 'hourly_gas_burn',
       'availability_mw', 'utilization', 'load', 'net_load', 'wind',
       'temperature', 'wind_speed', 'total_outages', 'load_actual',
       'net_load_actual', 'wind_actual', 'temperature_actual',
       'wind_speed_actual', 'total_outages_actual', 'gas_per_mw', 'load_final',
       'wind_final', 'temperature_final', 'hour', 'day_of_week', 'month',
       'gas_lag_1', 'gas_lag_24', 'gas_roll_24'],
      dtype='str')

In [22]:
model_df[model_df['datetime'] >= '2026-08-01']

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,day_of_week,month,gas_lag_1,gas_lag_24,gas_roll_24,year,gas_hour,gas_day_of_week,gas_month,gas_year
17519,2026-08-01,2026-07-31,CGS,0.0,0.0,NaN,NaN,87.0,NaN,40428.0,...,5,8,NaN,NaN,NaN,2026,0,4,7,2026
35039,2026-08-01,2026-07-31,DCS,2535.0,169.0,NaN,NaN,297.0,NaN,40428.0,...,5,8,NaN,NaN,NaN,2026,0,4,7,2026
52559,2026-08-01,2026-07-31,GGS,0.0,0.0,NaN,NaN,95.0,NaN,40428.0,...,5,8,NaN,NaN,NaN,2026,0,4,7,2026
70079,2026-08-01,2026-07-31,LCS,1954.5,130.3,NaN,NaN,203.0,NaN,40428.0,...,5,8,NaN,NaN,NaN,2026,0,4,7,2026
87599,2026-08-01,2026-07-31,PGS,0.0,0.0,NaN,NaN,422.9,NaN,40428.0,...,5,8,NaN,NaN,NaN,2026,0,4,7,2026


In [4]:
#adding time features
model_df["hour"] = model_df["datetime"].dt.hour
model_df["day_of_week"] = model_df["datetime"].dt.dayofweek
model_df["month"] = model_df["datetime"].dt.month
model_df['year'] = model_df['datetime'].dt.year


#adding time features
model_df["gas_hour"] = model_df["gas_day"].dt.hour
model_df["gas_day_of_week"] = model_df["gas_day"].dt.dayofweek
model_df["gas_month"] = model_df["gas_day"].dt.month
model_df['gas_year'] = model_df['gas_day'].dt.year

In [5]:
## Caleb getting info for saved version of model_df
print(model_df.shape)
model_df.info()

(87600, 36)
<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 36 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              87600 non-null  datetime64[us]
 1   gas_day               87600 non-null  datetime64[us]
 2   site                  87600 non-null  str           
 3   daily_site_gen_mw     87600 non-null  float64       
 4   hourly_site_gen_mw    87600 non-null  float64       
 5   daily_gas_burn        86565 non-null  float64       
 6   hourly_gas_burn       69429 non-null  float64       
 7   availability_mw       87600 non-null  float64       
 8   utilization           72697 non-null  float64       
 9   load                  87590 non-null  float64       
 10  net_load              87590 non-null  float64       
 11  wind                  87590 non-null  float64       
 12  temperature           87590 non-null  float64       
 13  wind_speed     

## Caleb Model Fitting and Prep
Initial goal on 8/4/2026 is to get an output that Clarissa can then format. Accuracy isn't too important at this point in time. Hopefully, it's reasonable but we're not immediately replacing the current forecasting method, so we do have some time to evaluate and iterate

**Current Process (8/4/2026)**
- Target variable is hourly gas burn
    - *Note that some of the columns are calculated using the hourly gas burn column.* These columns should not be used when fitting the model
- Use 2024-08-01 through 2026-05-31 to train a model
    - This still has some blank values but it > 99.5% filled in with the exception of the lagged variables and hourly gas burn
    - I'm going to assume for now that missing hourly gas burn means zero gas burn
    - I might remove the lagged variables from the initial model fitting
- Use 2026-06-01 - 2026-06-07 to get output for brief evaluation, but mainly so Clarissa has something to format for a screenshot


In [6]:
# Training Data
print(model_df[(model_df['datetime'] >'2024-08-01') & (model_df['datetime'] < '2026-06-01')].info())
#model_df.describe()


<class 'pandas.DataFrame'>
Index: 80275 entries, 0 to 86134
Data columns (total 36 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              80275 non-null  datetime64[us]
 1   gas_day               80275 non-null  datetime64[us]
 2   site                  80275 non-null  str           
 3   daily_site_gen_mw     80275 non-null  float64       
 4   hourly_site_gen_mw    80275 non-null  float64       
 5   daily_gas_burn        80275 non-null  float64       
 6   hourly_gas_burn       64637 non-null  float64       
 7   availability_mw       80275 non-null  float64       
 8   utilization           69866 non-null  float64       
 9   load                  80265 non-null  float64       
 10  net_load              80265 non-null  float64       
 11  wind                  80265 non-null  float64       
 12  temperature           80265 non-null  float64       
 13  wind_speed            80265 non-

In [7]:
# Testing Data
print(model_df[(model_df['datetime'] >'2026-06-01') & (model_df['datetime'] < '2026-06-08')].info())

<class 'pandas.DataFrame'>
Index: 835 entries, 16056 to 86302
Data columns (total 36 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              835 non-null    datetime64[us]
 1   gas_day               835 non-null    datetime64[us]
 2   site                  835 non-null    str           
 3   daily_site_gen_mw     835 non-null    float64       
 4   hourly_site_gen_mw    835 non-null    float64       
 5   daily_gas_burn        835 non-null    float64       
 6   hourly_gas_burn       606 non-null    float64       
 7   availability_mw       835 non-null    float64       
 8   utilization           617 non-null    float64       
 9   load                  835 non-null    float64       
 10  net_load              835 non-null    float64       
 11  wind                  835 non-null    float64       
 12  temperature           835 non-null    float64       
 13  wind_speed            835 non-

In [8]:
model_df[(model_df['datetime'] >'2024-08-01') & (model_df['datetime'] < '2026-06-01')].info()

<class 'pandas.DataFrame'>
Index: 80275 entries, 0 to 86134
Data columns (total 36 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              80275 non-null  datetime64[us]
 1   gas_day               80275 non-null  datetime64[us]
 2   site                  80275 non-null  str           
 3   daily_site_gen_mw     80275 non-null  float64       
 4   hourly_site_gen_mw    80275 non-null  float64       
 5   daily_gas_burn        80275 non-null  float64       
 6   hourly_gas_burn       64637 non-null  float64       
 7   availability_mw       80275 non-null  float64       
 8   utilization           69866 non-null  float64       
 9   load                  80265 non-null  float64       
 10  net_load              80265 non-null  float64       
 11  wind                  80265 non-null  float64       
 12  temperature           80265 non-null  float64       
 13  wind_speed            80265 non-

In [24]:
#Caleb's model fitting
#train models per site
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error


In [23]:
train_df = model_df[(model_df['datetime'] >= '2024-08-01') & (model_df['datetime'] < '2026-06-01')].sort_values(by = ['site', 'datetime'])

#X_train = train_df.loc[:, ['datetime', 'gas_day', 'gas_year', 'gas_month', 'gas_day', 'gas_hour', 'site', 'availability_mw', 'wind', 'temperature']]
X_train = train_df.loc[:, ['datetime', 'gas_day', 'year', 'month', 'day_of_week', 'hour', 'site', 'wind', 'temperature', 'wind_speed']]
X_train.info()

<class 'pandas.DataFrame'>
Index: 80275 entries, 0 to 86134
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   datetime     80275 non-null  datetime64[us]
 1   gas_day      80275 non-null  datetime64[us]
 2   year         80275 non-null  int32         
 3   month        80275 non-null  int32         
 4   day_of_week  80275 non-null  int32         
 5   hour         80275 non-null  int32         
 6   site         80275 non-null  str           
 7   wind         80265 non-null  float64       
 8   temperature  80265 non-null  float64       
 9   wind_speed   80265 non-null  float64       
dtypes: datetime64[us](2), float64(3), int32(4), str(1)
memory usage: 5.8 MB


In [25]:
y_train = train_df.loc[:, ['site', 'hourly_gas_burn']]

# Temporary fill to avoid errors. Need to confirm if blanks are equivalent to 0
y_train = y_train.fillna(0)
y_train.info()

<class 'pandas.DataFrame'>
Index: 80275 entries, 0 to 86134
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   site             80275 non-null  str    
 1   hourly_gas_burn  80275 non-null  float64
dtypes: float64(1), str(1)
memory usage: 2.1 MB


In [ ]:
test_df = model_df[(model_df['datetime'] >='2026-06-01') & (model_df['datetime'] < '2026-06-08')]

X_test = test_df.loc[:, ['datetime', 'gas_day', 'year', 'month', 'day_of_week', 'hour', 'site', 'wind', 'temperature', 'wind_speed']]

y_test = test_df.loc[:, ['site', 'hourly_gas_burn']]
y_test = y_test.fillna(0) # Temporary fill to avoid errors. Need to confirm if blanks are equivalent to 0

In [16]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

In [17]:
models = {}
results = {}
site_hourly_predictions ={}

sites = model_df["site"].unique()

for site in sites:

    print(f"\nTraining model for: {site}")

    X_train_site = X_train[X_train["site"] == site].drop(['site', 'datetime', 'gas_day'], axis=1)
    y_train_site = y_train[y_train['site']== site]['hourly_gas_burn']

    X_test_site = X_test[X_test["site"] == site].drop(['site', 'datetime', 'gas_day'], axis=1)
    y_test_site = y_test[y_test['site'] == site]['hourly_gas_burn']
    # CALEB NOTE: Rethink how we split this
    # split_idx = int(len(site_df) * 0.8)

    #train = site_df.iloc[:split_idx]
    #test  = site_df.iloc[split_idx:]
    print(X_test.head())

    # print(site, "X_train:", X_train_site.shape, "y_train:", y_train_site.shape)
    # print(site, "X_test:", X_test_site.shape, "y_train:", y_test_site.shape)
    
    
    # CALEB NOTE: Should we do some type of tuning??
    model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)

    model.fit(X_train_site, y_train_site)

    preds = model.predict(X_test_site)
    mae = mean_absolute_error(y_test_site, preds)

    #print(f"{site} MAE:", mae)

    #models[site] = model
    results[site] = mae
    sites_df = pd.DataFrame.from_dict({
            'datetime': X_test[X_test["site"] == site]['datetime'],
            'HE': "HE" + (X_test[X_test["site"] == site]['datetime'].dt.hour + 1).map(lambda x:f"{x:02d}"),
            'gas_day': X_test[X_test["site"] == site]['gas_day'],
            'site': site,
            'predicted_gas_burn': preds,
            'actual_gas_burn': y_test_site.values
        })

    site_hourly_predictions[site] = sites_df


Training model for: CGS
                 datetime    gas_day  year  month  day_of_week  hour site
16055 2026-06-01 00:00:00 2026-05-31  2026      6            0     0  CGS
16056 2026-06-01 01:00:00 2026-05-31  2026      6            0     1  CGS
16057 2026-06-01 02:00:00 2026-05-31  2026      6            0     2  CGS
16058 2026-06-01 03:00:00 2026-05-31  2026      6            0     3  CGS
16059 2026-06-01 04:00:00 2026-05-31  2026      6            0     4  CGS

Training model for: DCS
                 datetime    gas_day  year  month  day_of_week  hour site
16055 2026-06-01 00:00:00 2026-05-31  2026      6            0     0  CGS
16056 2026-06-01 01:00:00 2026-05-31  2026      6            0     1  CGS
16057 2026-06-01 02:00:00 2026-05-31  2026      6            0     2  CGS
16058 2026-06-01 03:00:00 2026-05-31  2026      6            0     3  CGS
16059 2026-06-01 04:00:00 2026-05-31  2026      6            0     4  CGS

Training model for: GGS
                 datetime    gas_day 

In [120]:
site_hourly_predictions

{'CGS':                  datetime    HE    gas_day site  predicted_gas_burn  \
 16055 2026-06-01 00:00:00  HE01 2026-05-31  CGS           -2.187379   
 16056 2026-06-01 01:00:00  HE02 2026-05-31  CGS           -3.765104   
 16057 2026-06-01 02:00:00  HE03 2026-05-31  CGS           -3.877537   
 16058 2026-06-01 03:00:00  HE04 2026-05-31  CGS            2.702576   
 16059 2026-06-01 04:00:00  HE05 2026-05-31  CGS            2.601079   
 ...                   ...   ...        ...  ...                 ...   
 16218 2026-06-07 19:00:00  HE20 2026-06-07  CGS           75.588448   
 16219 2026-06-07 20:00:00  HE21 2026-06-07  CGS           54.330357   
 16220 2026-06-07 21:00:00  HE22 2026-06-07  CGS           87.481926   
 16221 2026-06-07 22:00:00  HE23 2026-06-07  CGS           46.198170   
 16222 2026-06-07 23:00:00  HE24 2026-06-07  CGS           49.308540   
 
        actual_gas_burn  
 16055              0.0  
 16056              0.0  
 16057              0.0  
 16058              0.0

In [18]:
with pd.ExcelWriter('./output-data/Next Day Gasburn.xlsx', mode='w', engine='openpyxl') as file:
    for site in sites:
        df_tmp = site_hourly_predictions[site]
        #print(f"Site {site} has {len(df_tmp)} rows of data.")
        df_tmp.to_excel(file, sheet_name=site, index=False)
    

In [117]:
sites_df['site']

86135    PGS
86136    PGS
86137    PGS
86138    PGS
86139    PGS
        ... 
86298    PGS
86299    PGS
86300    PGS
86301    PGS
86302    PGS
Name: site, Length: 168, dtype: str

In [ ]:
site_df['CGS']

{'datetime': 16055   2026-06-01 00:00:00
 16056   2026-06-01 01:00:00
 16057   2026-06-01 02:00:00
 16058   2026-06-01 03:00:00
 16059   2026-06-01 04:00:00
                 ...        
 86298   2026-06-07 19:00:00
 86299   2026-06-07 20:00:00
 86300   2026-06-07 21:00:00
 86301   2026-06-07 22:00:00
 86302   2026-06-07 23:00:00
 Name: datetime, Length: 840, dtype: datetime64[us],
 'HE': 16055     0
 16056     1
 16057     2
 16058     3
 16059     4
          ..
 86298    19
 86299    20
 86300    21
 86301    22
 86302    23
 Name: datetime, Length: 840, dtype: int32,
 'gas_day':          gas_day    gas_day
 16055 2026-05-31 2026-05-31
 16056 2026-05-31 2026-05-31
 16057 2026-05-31 2026-05-31
 16058 2026-05-31 2026-05-31
 16059 2026-05-31 2026-05-31
 ...          ...        ...
 86298 2026-06-07 2026-06-07
 86299 2026-06-07 2026-06-07
 86300 2026-06-07 2026-06-07
 86301 2026-06-07 2026-06-07
 86302 2026-06-07 2026-06-07
 
 [840 rows x 2 columns],
 'site': 'PGS',
 'predicted_gas_burn'

In [ ]:
#Clarissa's model fitting
#train models per site
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

models = {}
results = {}

split_date = "2025-01-01"

for site in model_df["site"].unique():

    print(f"\nTraining model for: {site}")

    site_df = model_df[model_df["site"] == site].sort_values("datetime")

    # CALEB NOTE: Rethink how we split this
    split_idx = int(len(site_df) * 0.8)

    train = site_df.iloc[:split_idx]
    test  = site_df.iloc[split_idx:]


    print(site, "train:", len(train), "test:", len(test))
    
    
    X_train = train[features]
    y_train = train["hourly_gas_burn"]

    X_test = test[features]
    y_test = test["hourly_gas_burn"]

    # CALEB NOTE: Should we do some type of hyperparameter tuning??
    model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)

    print(f"{site} MAE:", mae)

    models[site] = model
    results[site] = mae



Training model for: CGS
CGS train: 4784 test: 1197
CGS MAE: 46.342156170134295

Training model for: DCS
DCS train: 5274 test: 1319
DCS MAE: 178.15525743592417

Training model for: GGS
GGS train: 396 test: 100
GGS MAE: 198.24632427250202

Training model for: LCS
LCS train: 9280 test: 2321
LCS MAE: 139.2352146306782

Training model for: PGS
PGS train: 12216 test: 3055
PGS MAE: 504.66481594138975


In [ ]:
## CALEB CHECK
models['CGS'].feature_names_in_

array(['load_final', 'wind_final', 'temperature_final', 'availability_mw',
       'utilization', 'hour', 'day_of_week', 'month', 'gas_lag_1',
       'gas_lag_24', 'gas_roll_24'], dtype='<U17')

In [ ]:
# back test validation of the forecast model 
for site in models:
    site_df = model_df[model_df["site"] == site]
    preds = models[site].predict(site_df[features])
    
    mae = mean_absolute_error(site_df["hourly_gas_burn"], preds)
    print(f"{site} Backtest MAE:", mae)

## Caleb notes: I believe units are MMBtu. I also have no idea if this is good or not

CGS Backtest MAE: 27.00908448291614
DCS Backtest MAE: 112.5304316860713
GGS Backtest MAE: 48.58178239485004
LCS Backtest MAE: 84.13669716306546
PGS Backtest MAE: 223.58049803377654


In [ ]:
# feature /drivers
for site in models:
    importances = pd.Series(models[site].feature_importances_, index=features).sort_values(ascending=False)

    print(f"\nTop drivers for {site}")
    print(importances.head(10))

## Caleb note: gas_lag_1 is winner and it's not really close other than for GGS



Top drivers for CGS
gas_lag_1            0.653287
hour                 0.084077
month                0.051822
gas_roll_24          0.050447
gas_lag_24           0.048039
temperature_final    0.033815
load_final           0.029092
day_of_week          0.025335
wind_final           0.024085
availability_mw      0.000000
dtype: float32

Top drivers for DCS
gas_lag_1            0.717173
hour                 0.050710
temperature_final    0.039128
month                0.038082
wind_final           0.037901
gas_lag_24           0.033693
day_of_week          0.031864
gas_roll_24          0.026854
load_final           0.024594
availability_mw      0.000000
dtype: float32

Top drivers for GGS
gas_lag_1            0.302872
month                0.136191
day_of_week          0.130401
temperature_final    0.111852
gas_roll_24          0.088105
gas_lag_24           0.074526
load_final           0.063876
wind_final           0.055728
hour                 0.036448
availability_mw      0.000000
dtype: 

In [126]:
print("Rows in master_df:", len(master_df))
print("Rows in model_df:", len(model_df))
print("Unique sites:", model_df["site"].unique())

## Caleb Note: There's a lot of information that got dropped going from master_df to model_df...probably not a good thing!!!

Rows in master_df: 86565
Rows in model_df: 39942
Unique sites: <ArrowStringArray>
['CGS', 'DCS', 'GGS', 'LCS', 'PGS']
Length: 5, dtype: str


In [ ]:
#create future timestamp

horizon = 168

now = pd.Timestamp.now()


## CALEB NOTE: Should this also be shifted to use `hours=10`?
forecast_start = ((now - pd.Timedelta(hours=9)).normalize() + pd.Timedelta(hours=9))

future_dates = pd.date_range(start=forecast_start, periods=horizon, freq="h")

In [ ]:
## CALEB CHECK: Has one week worth of data 168 hours = 7 full days
future_dates

DatetimeIndex(['2026-08-03 09:00:00', '2026-08-03 10:00:00',
               '2026-08-03 11:00:00', '2026-08-03 12:00:00',
               '2026-08-03 13:00:00', '2026-08-03 14:00:00',
               '2026-08-03 15:00:00', '2026-08-03 16:00:00',
               '2026-08-03 17:00:00', '2026-08-03 18:00:00',
               ...
               '2026-08-09 23:00:00', '2026-08-10 00:00:00',
               '2026-08-10 01:00:00', '2026-08-10 02:00:00',
               '2026-08-10 03:00:00', '2026-08-10 04:00:00',
               '2026-08-10 05:00:00', '2026-08-10 06:00:00',
               '2026-08-10 07:00:00', '2026-08-10 08:00:00'],
              dtype='datetime64[us]', length=168, freq='h')

In [ ]:
#ADD in forward looking data
#build future_df 

sites = model_df["site"].unique()


## CALEB NOTE: Cartesian product of future dates and sites
future_df = pd.MultiIndex.from_product([future_dates, sites], names=["datetime","site"]).to_frame(index=False)

In [134]:
future_df.info() #840 rows = 168 hrs * 5 sites

<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  840 non-null    datetime64[us]
 1   site      840 non-null    str           
dtypes: datetime64[us](1), str(1)
memory usage: 15.8 KB


In [135]:
future_df["hour"] = future_df["datetime"].dt.hour
future_df["day_of_week"] = future_df["datetime"].dt.dayofweek
future_df["month"] = future_df["datetime"].dt.month


In [ ]:
### Caleb Check
print(future_df.info())
future_df.head()
# for day of week monday=0

<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   datetime     840 non-null    datetime64[us]
 1   site         840 non-null    str           
 2   hour         840 non-null    int32         
 3   day_of_week  840 non-null    int32         
 4   month        840 non-null    int32         
dtypes: datetime64[us](1), int32(3), str(1)
memory usage: 25.7 KB
None


,datetime,site,hour,day_of_week,month
0,2026-08-03 09:00:00,CGS,9,0,8
1,2026-08-03 09:00:00,DCS,9,0,8
2,2026-08-03 09:00:00,GGS,9,0,8
3,2026-08-03 09:00:00,LCS,9,0,8
4,2026-08-03 09:00:00,PGS,9,0,8


In [139]:
# load and process availability
import os
from glob import glob


In [140]:

forward_folder = r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Transposed Forward Looking Data"

print("Folder exists:", os.path.exists(forward_folder))


Folder exists: True


In [141]:
try:
    print("Directory contents:")
    for f in os.listdir(forward_folder):
        print(f)
except Exception as e:
    print("Error accessing folder:", e)


Directory contents:
excel forward looking transposed (6-26 thru 7-03).xlsx
excel forward looking transposed (6-29 thru 7-06).xlsx
excel forward looking transposed (6-30 thru 7-07).xlsx
excel forward looking transposed (7-02 thru 7-09).xlsx
excel forward looking transposed (7-07 thru 7-14).xlsx
excel forward looking transposed (7-09 thru 7-16).xlsx
excel forward looking transposed (7-10 thru 7-17).xlsx
excel forward looking transposed (7-13 thru 7-20).xlsx
excel forward looking transposed (7-17 thru 7-24).xlsx
excel forward looking transposed (7-27 thru 8-3).xlsx
excel forward looking transposed (7-28 thru 7-3).xlsx
~$excel forward looking transposed (7-27 thru 8-3).xlsx


In [142]:
# getting the forward looking HEL
files = [ f for f in glob(os.path.join(forward_folder, "*forward*transposed*.xls*"))
    if not os.path.basename(f).startswith("~$")]


if not files:
    raise FileNotFoundError("No forward-looking files matched pattern")


In [143]:
files

['G:\\Trading\\Forecasts\\Daily Gas Burn Forecast by Site\\Transposed Forward Looking Data\\excel forward looking transposed (6-26 thru 7-03).xlsx',
 'G:\\Trading\\Forecasts\\Daily Gas Burn Forecast by Site\\Transposed Forward Looking Data\\excel forward looking transposed (6-29 thru 7-06).xlsx',
 'G:\\Trading\\Forecasts\\Daily Gas Burn Forecast by Site\\Transposed Forward Looking Data\\excel forward looking transposed (6-30 thru 7-07).xlsx',
 'G:\\Trading\\Forecasts\\Daily Gas Burn Forecast by Site\\Transposed Forward Looking Data\\excel forward looking transposed (7-02 thru 7-09).xlsx',
 'G:\\Trading\\Forecasts\\Daily Gas Burn Forecast by Site\\Transposed Forward Looking Data\\excel forward looking transposed (7-07 thru 7-14).xlsx',
 'G:\\Trading\\Forecasts\\Daily Gas Burn Forecast by Site\\Transposed Forward Looking Data\\excel forward looking transposed (7-09 thru 7-16).xlsx',
 'G:\\Trading\\Forecasts\\Daily Gas Burn Forecast by Site\\Transposed Forward Looking Data\\excel forward 

In [ ]:

# grab the most recent file
forward_file = max(files, key=os.path.getctime)


## CALEB NOTE: Better when needing to read multiple sheets
xls = pd.ExcelFile(forward_file)

sheet_name = [s for s in xls.sheet_names if "transposed" in s.lower()][0]

forward_df = pd.read_excel(xls, sheet_name=sheet_name)
#forward_df is the 'Transposed' sheet (rows = datetimes, columns are units)
#forward also has an extra day (192 rows => 8 days) when compared to future_df (168 rows => 7 days)
#looks like datetime is coming in as an excel date time (ie. as a float). Next cell has code to properly convert to a pandas datetime


In [156]:
# Unit and Origin are needed to convert from the excel style float values where 0 is equal to 1900-01-01
pd.to_datetime(forward_df['datetime'], unit='D', origin='1899-12-30').dt.round('h')

0     2026-07-28 00:00:00
1     2026-07-28 01:00:00
2     2026-07-28 02:00:00
3     2026-07-28 03:00:00
4     2026-07-28 04:00:00
              ...        
187   2026-08-04 19:00:00
188   2026-08-04 20:00:00
189   2026-08-04 21:00:00
190   2026-08-04 22:00:00
191   2026-08-04 23:00:00
Name: datetime, Length: 192, dtype: datetime64[ns]

In [ ]:
# CALEB CHECK
forward_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   datetime                          192 non-null    float64
 1       CGS1 - High Effective Limit   192 non-null    int64  
 2       DCS1 - High Effective Limit   192 non-null    int64  
 3       GGS1 - High Effective Limit   192 non-null    int64  
 4       GGS2 - High Effective Limit   192 non-null    int64  
 5       LCS1 - High Effective Limit   192 non-null    int64  
 6       LCS2 - High Effective Limit   192 non-null    int64  
 7       LCS3 - High Effective Limit   192 non-null    int64  
 8       LCS4 - High Effective Limit   192 non-null    int64  
 9       LCS5 - High Effective Limit   192 non-null    int64  
 10      LCS6 - High Effective Limit   192 non-null    int64  
 11      PGS1 - High Effective Limit   192 non-null    int64  
 12      PGS2 - High Eff

In [157]:
## CALEB CHECK
forward_df.head()

,datetime,CGS1 - High Effective Limit,DCS1 - High Effective Limit,GGS1 - High Effective Limit,GGS2 - High Effective Limit,LCS1 - High Effective Limit,LCS2 - High Effective Limit,LCS3 - High Effective Limit,LCS4 - High Effective Limit,LCS5 - High Effective Limit,...,PGS21 - High Effective Limit,PGS22 - High Effective Limit,PGS31 - High Effective Limit,PGS32 - High Effective Limit,PGS33 - High Effective Limit,PGS34 - High Effective Limit,PGS35 - High Effective Limit,PGS36 - High Effective Limit,PGS4 - High Effective Limit,PGS5 - High Effective Limit
0,46231.000000,0,297,71,62,32,40,40,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,216.72,216.72
1,46231.041667,0,297,61,62,32,40,40,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,217.60,217.60
2,46231.083333,0,297,61,82,32,41,41,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,218.50,218.50
3,46231.125000,0,297,61,82,32,41,41,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,218.78,218.78
4,46231.166667,0,297,58,82,32,41,41,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,216.34,216.34


In [158]:
# cleaning the file
forward_df.columns = [col.replace(" - High Effective Limit", "").strip()
    for col in forward_df.columns]

forward_df["datetime"] = pd.to_datetime(forward_df["datetime"])


In [ ]:
## CALEB CHECK
## Note that the datetimes are in the 1970s, because the excel time shift wasn't accounted for

print(f'dimensions of forward_df: {forward_df.shape}')
forward_df.head()

dimensions of forward_df: (192, 34)


,datetime,CGS1,DCS1,GGS1,GGS2,LCS1,LCS2,LCS3,LCS4,LCS5,...,PGS21,PGS22,PGS31,PGS32,PGS33,PGS34,PGS35,PGS36,PGS4,PGS5
0,1970-01-01 00:00:00.000046231,0,297,71,62,32,40,40,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,216.72,216.72
1,1970-01-01 00:00:00.000046231,0,297,61,62,32,40,40,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,217.60,217.60
2,1970-01-01 00:00:00.000046231,0,297,61,82,32,41,41,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,218.50,218.50
3,1970-01-01 00:00:00.000046231,0,297,61,82,32,41,41,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,218.78,218.78
4,1970-01-01 00:00:00.000046231,0,297,58,82,32,41,41,38,37,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,216.34,216.34


In [161]:
# melting data into a format that can be used
forward_long = forward_df.melt( id_vars=["datetime"], var_name="unit", value_name="availability_mw")


In [166]:
## CALEB CHECK
print(f'dimensions of forward_long: {forward_long.shape}')
forward_long[forward_long['unit']=='DCS1'].head()

dimensions of forward_long: (6336, 3)


,datetime,unit,availability_mw
192,1970-01-01 00:00:00.000046231,DCS1,297.0
193,1970-01-01 00:00:00.000046231,DCS1,297.0
194,1970-01-01 00:00:00.000046231,DCS1,297.0
195,1970-01-01 00:00:00.000046231,DCS1,297.0
196,1970-01-01 00:00:00.000046231,DCS1,297.0


In [167]:
def map_unit_to_site(u):
    if str(u).startswith("DCS"):
        return "DCS"
    elif str(u).startswith("LCS"):
        return "LCS"
    elif str(u).startswith("PGS"):
        return "PGS"
    elif str(u).startswith("GGS"):
        return "GGS"
    elif str(u).startswith("CGS"):
        return "CGS"
    return None


In [168]:
forward_long["site"] = forward_long["unit"].apply(map_unit_to_site)

In [ ]:
## Caleb Check on nulls
forward_long['site'].isna().sum()

np.int64(0)

In [170]:
forward_long = forward_long.dropna(subset=["site"])


In [178]:
## Caleb Check
forward_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 6336 entries, 0 to 6335
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   datetime         6336 non-null   datetime64[ns]
 1   unit             6336 non-null   str           
 2   availability_mw  6336 non-null   float64       
 3   site             6336 non-null   str           
dtypes: datetime64[ns](1), float64(1), str(2)
memory usage: 245.6 KB


In [181]:
# Caleb Check
forward_long[forward_long['availability_mw'] > 0]

,datetime,unit,availability_mw,site
172,1970-01-01 00:00:00.000046238,CGS1,95.00,CGS
173,1970-01-01 00:00:00.000046238,CGS1,95.00,CGS
174,1970-01-01 00:00:00.000046238,CGS1,95.00,CGS
175,1970-01-01 00:00:00.000046238,CGS1,95.00,CGS
176,1970-01-01 00:00:00.000046238,CGS1,95.00,CGS
...,...,...,...,...
6331,1970-01-01 00:00:00.000046238,PGS5,218.77,PGS
6332,1970-01-01 00:00:00.000046238,PGS5,219.55,PGS
6333,1970-01-01 00:00:00.000046238,PGS5,220.27,PGS
6334,1970-01-01 00:00:00.000046238,PGS5,221.60,PGS


In [171]:
forward_site = (forward_long.groupby(["datetime","site"], as_index=False)["availability_mw"].sum())


In [172]:
# merge availability 
future_df = future_df.merge(forward_site, on=["datetime","site"], how="left")

#caleb note: chaining ffill and bfill to get missing at beginning and end of dataset
future_df["availability_mw"] = (future_df["availability_mw"].ffill().bfill())

In [174]:
# add utilization
util_lookup = master_df.groupby(["site","hour"])["utilization"].mean()

future_df = future_df.merge(util_lookup.rename("utilization"), on=["site","hour"], how="left")

In [177]:
future_df

,datetime,site,hour,day_of_week,month,availability_mw,utilization_x,utilization_y
0,2026-08-03 09:00:00,CGS,9,0,8,NaN,NaN,NaN
1,2026-08-03 09:00:00,DCS,9,0,8,NaN,NaN,NaN
2,2026-08-03 09:00:00,GGS,9,0,8,NaN,NaN,NaN
3,2026-08-03 09:00:00,LCS,9,0,8,NaN,NaN,NaN
4,2026-08-03 09:00:00,PGS,9,0,8,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
835,2026-08-10 08:00:00,CGS,8,0,8,NaN,NaN,NaN
836,2026-08-10 08:00:00,DCS,8,0,8,NaN,NaN,NaN
837,2026-08-10 08:00:00,GGS,8,0,8,NaN,NaN,NaN
838,2026-08-10 08:00:00,LCS,8,0,8,NaN,NaN,NaN


In [182]:
# merge forecast data
## CALEB NOTE: what is this df_forecast_clean object? I don't see it defined anywhere???
future_df = future_df.merge(df_forecast, on="datetime", how="left")


In [183]:
# cleaning up the naming 
def safe_final(df, actual, forecast):
    if actual in df.columns:
        return df[actual].fillna(df[forecast])
    return df[forecast]

future_df["load_final"] = safe_final(future_df, "load_actual", "load")
future_df["wind_final"] = safe_final(future_df, "wind_actual", "wind")
future_df["temperature_final"] = safe_final(future_df, "temperature_actual", "temperature")


In [185]:
############################################
### CALEB CHECK HERE #######################
############################################
#building forecasting loop
#add site capacity CHECK THESE NUMBERS
site_capacity = {"CGS": 95, "DCS": 297, "LCS": 270, "PGS": 803.4, "GGS": 190}

gas_per_mw_lookup = master_df.groupby(["site", "hour"])["gas_per_mw"].median()


In [ ]:

# forecasting loop
all_forecasts = []

for site in models.keys():

    print(f"\nForecasting for {site}")

    model = models[site]

    site_hist = master_df[master_df["site"] == site].sort_values("datetime")
    site_hist = site_hist[site_hist["datetime"] < future_dates.min()]

    history = site_hist.tail(48).copy()
    check1 = history.shape # delete later
    future_preds = []

    dt_row = -1
    for dt in future_dates:
        dt_row += 1
        # 
        new_row = { "datetime": dt, "site": site, "hour": dt.hour, "day_of_week": dt.dayofweek, "month": dt.month}

        api_row = future_df[
            (future_df["datetime"] == dt) & 
            (future_df["site"] == site)]

        # forecast inputs
        if len(api_row) > 0:
            new_row["load_final"] = api_row["load_final"].values[0]
            new_row["wind_final"] = api_row["wind_final"].values[0]
            new_row["temperature_final"] = api_row["temperature_final"].values[0]
            new_row["availability_mw"] = api_row["availability_mw"].values[0]
        else:
            #Caleb assumption is that .iloc should really be .values[-1]?
            new_row["load_final"] = history["load_final"].iloc[-1]
            new_row["wind_final"] = history["wind_final"].iloc[-1]
            new_row["temperature_final"] = history["temperature_final"].iloc[-1]
            new_row["availability_mw"] = history["availability_mw"].iloc[-1]


        # utilization
        lookup_val = util_lookup.get((site, dt.hour), np.nan)

        if not pd.isna(lookup_val):
            new_row["utilization"] = lookup_val
        else:
            # Purpose
            # check2=history.shape
            #new_row["utilization"] = history["utilization"].dropna().iloc[-1]
            new_row["utilization"] = history["utilization"]..iloc[-1]
            #look into this as well 
            # lags
            new_row["gas_lag_1"] = history["hourly_gas_burn"].iloc[-1]
            new_row["gas_lag_24"] = history["hourly_gas_burn"].iloc[-24]
            new_row["gas_roll_24"] = history["hourly_gas_burn"].iloc[-24:].mean()

        # building the input
        X_pred = pd.DataFrame([new_row])[features]

        # prediction
        pred = model.predict(X_pred)[0]

        # availability constraint
        cap = site_capacity.get(site, 200)

        if new_row["availability_mw"] <= 1:
            pred = 0
        else:
            scale = min(new_row["availability_mw"] / cap, 1)
            pred *= scale





#something to look into 
        # minimum mw constraint
        gas_per_mw = gas_per_mw_lookup.get((site, dt.hour), 8)
        implied_mw = pred / gas_per_mw

        if implied_mw < 0:
            pred = 0

        new_row["predicted_gas_burn"] = pred

        # recursion
        history = pd.concat(
            [history, pd.DataFrame([{**new_row, "hourly_gas_burn": pred}])], ignore_index=True)

        future_preds.append(new_row)

    all_forecasts.append(pd.DataFrame(future_preds))



Forecasting for CGS


IndexError: single positional indexer is out-of-bounds

In [ ]:





    
    
    

    #combine all sites
forecast_df = pd.concat(all_forecasts, ignore_index=True)



# adding gas day
forecast_df["gas_day"] = (pd.to_datetime(forecast_df["datetime"]) - pd.Timedelta(hours=9)).dt.date

# and gas_per_mw lookup 
gas_per_mw_lookup = master_df.groupby(["site", "hour"])["gas_per_mw"].median()

# add hour for merge 
forecast_df["hour"] = pd.to_datetime(forecast_df["datetime"]).dt.hour

# then merge gas_per_mw 
forecast_df = forecast_df.merge(gas_per_mw_lookup.rename("gas_per_mw"), on=["site", "hour"], how="left")

forecast_df["implied_mw"] = forecast_df["predicted_gas_burn"] / forecast_df["gas_per_mw"]



# final formatting here
forecast_df = forecast_df[["datetime", "gas_day", "site", "predicted_gas_burn", "gas_per_mw", "implied_mw"]]

# now daily aggregation
daily_forecast = (forecast_df.groupby(["gas_day", "site"], as_index=False)["predicted_gas_burn"].sum())

# and sort + format 
forecast_df = forecast_df.sort_values(["site", "datetime"])







#final formatting:
forecast_df = forecast_df.drop_duplicates(subset=["datetime", "site"])
forecast_df = forecast_df.sort_values(["site", "datetime"])
forecast_df["datetime"] = forecast_df["datetime"].dt.strftime("%Y-%m-%d %H:%M")


#export to excel
file_path = r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\forecast with historicals checked.xlsx"

with pd.ExcelWriter(file_path, engine="openpyxl") as writer:

    for site in forecast_df["site"].unique():
        site_df = forecast_df[forecast_df["site"] == site]
        site_df.to_excel(writer, sheet_name=site, index=False)

print("File saved at:", file_path)